# Intelligent Email Forensics Using Deep Learning and RAG
**One notebook, end to end:** data -> DistilBERT classifier -> RAG (dense + BM25 + re-ranker) -> explainability -> header forensics -> advanced forensic features -> web front end.

**How to run (Google Colab)**
1. *Runtime* -> *Change runtime type* -> **T4 GPU**
2. *Runtime* -> **Run all** (about 15-20 min in the default FAST_MODE). Keep the tab open.
3. The last-but-one cell launches the **web app** and prints a public link (`https://....gradio.live`).
4. Do not run cells out of order, and do not run the check/save cells before the training cells have finished.

| Part | Contents |
|---|---|
| A | Core pipeline: data, indicators, DistilBERT, FAISS RAG, LLM report |
| B | Extensions: baselines, explainability, hybrid retrieval, RAG evaluation, header forensics |
| C | Advanced features: hybrid header+semantic vectors, hash-cited grounding and chain of custody, GraphRAG timeline, PII redaction, obfuscation decoding + OCR |
| D | Front end (Gradio) |
| E | Save results |

Settings you may want to change are in the config cell (`FAST_MODE`, `USE_LLM`, `RUN_ROBERTA`, `SAVE_TO_DRIVE`).

## Part A - Core pipeline
### Setup

In [ ]:
!pip install -q faiss-cpu sentence-transformers kagglehub gradio networkx pytesseract pillow spacy
!apt-get -qq install -y tesseract-ocr > /dev/null
!python -m spacy download en_core_web_sm -q

In [ ]:
# Step 0: download the dataset (public Kaggle dataset); falls back to manual upload
import os, shutil, kagglehub
os.makedirs("/content/data", exist_ok=True)
try:
    p = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")
    for f in os.listdir(p):
        if f.endswith(".csv"):
            shutil.copy(os.path.join(p, f), "/content/data")
except Exception as e:
    print("Auto-download failed:", e)
if not any(f.endswith(".csv") for f in os.listdir("/content/data")):
    print("Please upload the dataset CSV files now (CEAS_08.csv, Enron.csv, ...)")
    from google.colab import files
    for name, data in files.upload().items():
        open(f"/content/data/{name}", "wb").write(data)
print(os.listdir("/content/data"))

In [ ]:
import os, re, glob, warnings, random
import numpy as np, pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ---- Config ----
DATA_DIR  = "/content/data"
OUT_DIR   = "/content/outputs"
os.makedirs(OUT_DIR, exist_ok=True)
GPU = torch.cuda.is_available()
CLS_MODEL   = "distilbert-base-uncased"
EMB_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
FAST_MODE     = True    # True = ~15-20 min run; False = full 20,000 emails (~45 min)
USE_LLM       = True    # LLM-written reports (GPU only); False = template reports
RUN_ROBERTA   = False   # also fine-tune RoBERTa for the model comparison (+10-15 min)
SAVE_TO_DRIVE = False   # copy the trained model and results to Google Drive
if GPU:
    N_SAMPLES = 8000 if FAST_MODE else 20000
else:
    N_SAMPLES = 3000
MAX_LEN = 256 if GPU else 128
EPOCHS = 2 if GPU else 1
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if (GPU and USE_LLM) else None

## 1. Load and clean the data

In [ ]:
files = glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)
print("\n".join(files)); assert files, f"No CSV files found in {DATA_DIR} - download the dataset first"

frames = []
for f in files:
    try:
        d = pd.read_csv(f, on_bad_lines="skip", engine="python")
    except Exception as e:
        print("skip", f, e); continue
    d.columns = [c.strip().lower() for c in d.columns]
    if "body" in d.columns and "label" in d.columns:
        for c in ["sender", "receiver", "date", "subject", "urls"]:
            if c not in d.columns: d[c] = np.nan
        d["source"] = os.path.basename(f).replace(".csv", "")
        frames.append(d[["sender", "receiver", "date", "subject", "body", "urls", "label", "source"]])
    else:
        print("  -> skipped (no body/label):", os.path.basename(f), list(d.columns)[:6])

df = pd.concat(frames, ignore_index=True)
print(df.shape)
df.head()

In [ ]:
def clean_text(t):
    t = str(t)
    t = re.sub(r"<[^>]+>", " ", t)          # strip HTML tags
    t = re.sub(r"\s+", " ", t).strip()
    return t

# normalise labels to 0 = legitimate, 1 = phishing/spam
if df["label"].dtype == object:
    df["label"] = df["label"].astype(str).str.lower().map(
        lambda x: 1 if x in ("1", "spam", "phishing", "phishing email", "fraud") else 0)
df["label"] = df["label"].astype(int)

df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")
df["text"] = ("Subject: " + df["subject"].astype(str) + " | " + df["body"].map(clean_text))
df = df[df["text"].str.len() > 30].drop_duplicates(subset="text").reset_index(drop=True)

# balanced-ish sample
per_class = N_SAMPLES // 2
df = (df.groupby("label", group_keys=False)
        .apply(lambda g: g.sample(min(len(g), per_class), random_state=SEED))
        .sample(frac=1, random_state=SEED).reset_index(drop=True))
df["id"] = df.index

print(df["label"].value_counts())
sns.countplot(x="label", data=df); plt.title("Class balance (0=legit, 1=phish/spam)"); plt.show()

## 2. Forensic indicator extraction
Rule-based evidence that will later be shown in the forensic report next to the deep-learning verdict.

In [ ]:
URL_RE = re.compile(r"https?://[^\s\"'<>)]+", re.I)
IP_URL_RE = re.compile(r"https?://\d{1,3}(?:\.\d{1,3}){3}", re.I)
SHORTENERS = ("bit.ly", "tinyurl", "goo.gl", "t.co", "ow.ly", "is.gd", "buff.ly", "rebrand.ly")
URGENCY = ["urgent", "immediately", "within 24 hours", "act now", "suspended", "final notice",
           "verify your account", "expires", "limited time", "last warning", "action required"]
CREDENTIAL = ["password", "login", "log in", "sign in", "credentials", "ssn", "social security",
              "bank account", "credit card", "pin", "otp", "verification code"]
MONEY = ["wire transfer", "bitcoin", "gift card", "lottery", "inheritance", "million", "prize",
         "payment", "invoice", "refund", "western union"]

def forensic_indicators(text, sender=None):
    low = text.lower()
    urls = URL_RE.findall(text)
    ind = {
        "num_urls": len(urls),
        "ip_based_url": bool(IP_URL_RE.search(text)),
        "url_shortener": any(s in u.lower() for u in urls for s in SHORTENERS),
        "urgency_terms": [w for w in URGENCY if w in low],
        "credential_terms": [w for w in CREDENTIAL if w in low],
        "financial_terms": [w for w in MONEY if w in low],
        "exclamations": text.count("!"),
        "upper_ratio": round(sum(c.isupper() for c in text) / max(len(text), 1), 3),
        "has_html_form": "<form" in low,
    }
    if sender and isinstance(sender, str) and "@" in sender:
        ind["sender_domain"] = sender.split("@")[-1].strip("> ").lower()
    score = (ind["ip_based_url"] * 2 + ind["url_shortener"] * 1 + min(len(ind["urgency_terms"]), 3)
             + min(len(ind["credential_terms"]), 3) + min(len(ind["financial_terms"]), 2)
             + (ind["upper_ratio"] > 0.25))
    ind["rule_risk_score"] = int(score)
    return ind

print(forensic_indicators("URGENT! Verify your account at http://192.168.4.5/login within 24 hours or it will be suspended"))

## 3. Train / validation / test split

In [ ]:
train_df, tmp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=0.5, stratify=tmp_df["label"], random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

## 4. Fine-tune DistilBERT classifier

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)

tok = AutoTokenizer.from_pretrained(CLS_MODEL)

class EmailDS(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.enc = tok(frame["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.labels = frame["label"].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

train_ds, val_ds, test_ds = EmailDS(train_df), EmailDS(val_df), EmailDS(test_df)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="binary")
    return {"acc": accuracy_score(p.label_ids, preds), "precision": pr, "recall": rc, "f1": f1}

model = AutoModelForSequenceClassification.from_pretrained(CLS_MODEL, num_labels=2,
            id2label={0: "legitimate", 1: "phishing/spam"}, label2id={"legitimate": 0, "phishing/spam": 1})

args = TrainingArguments(
    output_dir=f"{OUT_DIR}/cls_ckpt",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()

In [ ]:
out = trainer.predict(test_ds)
y_pred = out.predictions.argmax(-1)
y_true = out.label_ids
print(classification_report(y_true, y_pred, target_names=["legitimate", "phishing/spam"], digits=4))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["legit", "phish"], yticklabels=["legit", "phish"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Test confusion matrix"); plt.show()

trainer.save_model(f"{OUT_DIR}/email_cls"); tok.save_pretrained(f"{OUT_DIR}/email_cls")

In [ ]:
clf = trainer.model.eval()

@torch.no_grad()
def classify(text):
    enc = tok(text, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(clf.device)
    probs = torch.softmax(clf(**enc).logits, -1)[0].cpu().numpy()
    return {"label": "phishing/spam" if probs[1] > 0.5 else "legitimate", "p_malicious": float(probs[1])}

print(classify("Dear customer, your account is suspended. Click http://bit.ly/x9 to verify your password now!"))

In [ ]:
# Optional: keep the trained model and results on Google Drive (set SAVE_TO_DRIVE = True in the config cell)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/email_forensics"
    os.makedirs(dest, exist_ok=True)
    shutil.copytree(f"{OUT_DIR}/email_cls", f"{dest}/email_cls", dirs_exist_ok=True)
    print("Model saved to", dest)
else:
    print("SAVE_TO_DRIVE is False - skipping")

## 5. RAG knowledge base (embeddings + FAISS)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)
kb = train_df.reset_index(drop=True)          # knowledge base = known (training) emails
kb_vecs = embedder.encode(kb["text"].tolist(), batch_size=128, show_progress_bar=True,
                          normalize_embeddings=True).astype("float32")

index = faiss.IndexFlatIP(kb_vecs.shape[1])    # cosine similarity (vectors are normalised)
index.add(kb_vecs)
print("Indexed:", index.ntotal)

def retrieve(query, k=5, label=None):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    sims, idx = index.search(q, k * 4 if label is not None else k)
    res = []
    for s, i in zip(sims[0], idx[0]):
        r = kb.iloc[i]
        if label is not None and r["label"] != label: continue
        res.append({"id": int(r["id"]), "similarity": float(s), "label": int(r["label"]),
                    "source": r["source"], "sender": r["sender"], "snippet": r["text"][:300]})
        if len(res) == k: break
    return res

for r in retrieve("Your PayPal account has been limited, confirm your identity", k=3):
    print(round(r["similarity"], 3), r["label"], r["snippet"][:120])

## 6. LLM for forensic report generation
If the LLM can't load (no internet / out of memory) the notebook automatically falls back to a template report.

In [ ]:
from transformers import AutoModelForCausalLM

llm, llm_tok = None, None
if LLM_MODEL:
    try:
        llm_tok = AutoTokenizer.from_pretrained(LLM_MODEL)
        llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE)
        print("LLM loaded:", LLM_MODEL)
    except Exception as e:
        print("LLM not available, using template reports ->", e)

SYSTEM = ("You are a digital-forensics analyst. Using ONLY the evidence provided (classifier output, rule-based "
          "indicators, and similar known emails), write a short, factual forensic report with: Verdict, Key Evidence, "
          "Similar Known Cases, Risk Level (Low/Medium/High) and Recommended Actions. Do not invent facts.")

def llm_generate(prompt, max_new_tokens=350):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    text = llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
def forensic_report(email_text, sender=None, k=3):
    cls = classify(email_text)
    ind = forensic_indicators(email_text, sender)
    sims = retrieve(email_text, k=k)
    risk = "High" if (cls["p_malicious"] > 0.85 or ind["rule_risk_score"] >= 5) else \
           "Medium" if (cls["p_malicious"] > 0.5 or ind["rule_risk_score"] >= 3) else "Low"

    evidence = (f"CLASSIFIER: {cls['label']} (p_malicious={cls['p_malicious']:.3f})\n"
                f"INDICATORS: {ind}\n"
                f"COMPUTED RISK LEVEL: {risk}\n"
                "SIMILAR KNOWN EMAILS:\n" +
                "\n".join(f"- [sim={s['similarity']:.2f}, {'MALICIOUS' if s['label'] else 'LEGIT'}, src={s['source']}] {s['snippet'][:200]}"
                          for s in sims))
    if llm is not None:
        narrative = llm_generate(f"EMAIL UNDER INVESTIGATION:\n{email_text[:1500]}\n\n{evidence}")
    else:
        narrative = (f"Verdict: {cls['label']} (confidence {cls['p_malicious']:.1%}). Risk: {risk}.\n"
                     f"Key evidence: {ind['num_urls']} URL(s); IP-based link={ind['ip_based_url']}; shortener={ind['url_shortener']}; "
                     f"urgency={ind['urgency_terms']}; credential bait={ind['credential_terms']}; financial={ind['financial_terms']}.\n"
                     f"Similar known cases: {sum(s['label'] for s in sims)}/{len(sims)} retrieved neighbours are malicious.")
    return {"classification": cls, "indicators": ind, "risk": risk, "similar_emails": sims, "report": narrative}

sample = test_df[test_df.label == 1].iloc[0]
res = forensic_report(sample["text"], sample["sender"])
print("TRUE LABEL:", sample["label"], "| RISK:", res["risk"], "\n")
print(res["report"])

## 7. Investigator Q&A over the corpus (RAG)

In [ ]:
def ask(question, k=5, label=None):
    ctx = retrieve(question, k=k, label=label)
    context = "\n".join(f"[{i+1}] ({'malicious' if c['label'] else 'legit'}, {c['source']}) {c['snippet']}" for i, c in enumerate(ctx))
    if llm is None:
        return context
    msgs = [{"role": "system", "content": "You are an email-forensics assistant. Answer only from the numbered email excerpts and cite them like [1]."},
            {"role": "user", "content": f"Excerpts:\n{context}\n\nQuestion: {question}"}]
    text = llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=300, do_sample=False)
    return llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask("What tactics do phishing emails use to pressure victims into clicking links?", label=1))

## 8. Evaluate the full pipeline on a few test emails

In [ ]:
rows = []
for _, r in test_df.sample(30, random_state=SEED).iterrows():
    cls = classify(r["text"]); ind = forensic_indicators(r["text"], r["sender"])
    rows.append({"true": r["label"], "pred": int(cls["label"] != "legitimate"),
                 "p_malicious": round(cls["p_malicious"], 3), "rule_score": ind["rule_risk_score"]})
pd.DataFrame(rows).head(15)

## Part B - Extensions

## 9. Baseline and model comparison

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
Xtr = tfidf.fit_transform(train_df["text"]); Xte = tfidf.transform(test_df["text"])
lr = LogisticRegression(max_iter=1000, C=4).fit(Xtr, train_df["label"])
p_lr = lr.predict(Xte)

def score_row(name, y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="binary")
    return {"model": name, "accuracy": accuracy_score(y, p), "precision": pr, "recall": rc, "f1": f1}

rows = [score_row("TF-IDF + LogReg", test_df["label"], p_lr),
        score_row("DistilBERT (fine-tuned)", y_true, y_pred)]
pd.DataFrame(rows).round(4)

In [ ]:
# Optional: fine-tune a second transformer for comparison (takes ~10-15 min on a T4)
# RUN_ROBERTA is set in the config cell

def train_hf(model_name, epochs=2, bs=16, lr_=2e-5):
    t = AutoTokenizer.from_pretrained(model_name)
    class DS(torch.utils.data.Dataset):
        def __init__(s, f): s.e = t(f["text"].tolist(), truncation=True, max_length=MAX_LEN); s.l = f["label"].tolist()
        def __len__(s): return len(s.l)
        def __getitem__(s, i):
            d = {k: v[i] for k, v in s.e.items()}; d["labels"] = s.l[i]; return d
    m = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    a = TrainingArguments(output_dir=f"{OUT_DIR}/tmp_{model_name.replace('/', '_')}", num_train_epochs=epochs,
                          per_device_train_batch_size=bs, per_device_eval_batch_size=64, learning_rate=lr_,
                          weight_decay=0.01, eval_strategy="no", save_strategy="no",
                          fp16=torch.cuda.is_available(), report_to="none", seed=SEED)
    tr = Trainer(model=m, args=a, train_dataset=DS(train_df), data_collator=DataCollatorWithPadding(t))
    tr.train()
    return tr.predict(DS(test_df)).predictions.argmax(-1)

if RUN_ROBERTA:
    p_rb = train_hf("roberta-base")
    rows.append(score_row("RoBERTa-base", test_df["label"], p_rb))

cmp = pd.DataFrame(rows).round(4)
display(cmp)
cmp.set_index("model")[["accuracy", "precision", "recall", "f1"]].plot.bar(figsize=(8, 4), ylim=(0.8, 1.0), rot=15)
plt.title("Model comparison (test set)"); plt.tight_layout(); plt.show()

## 10. Explainability - word-level occlusion
Each word window is removed in turn; the drop in the malicious probability shows how much that part pushed the verdict. Red = pushes toward phishing, blue = pushes toward legitimate.

In [ ]:
from IPython.display import HTML, display
import html as _html

@torch.no_grad()
def p_mal_batch(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(clf.device)
        out += torch.softmax(clf(**enc).logits, -1)[:, 1].cpu().tolist()
    return np.array(out)

def explain(text, max_words=100, window=2, show=True):
    words = text.split()[:max_words]
    n = len(words)
    base = p_mal_batch([" ".join(words)])[0]
    starts = list(range(0, max(n - window + 1, 1)))
    variants = [" ".join(words[:i] + words[i + window:]) for i in starts]
    drops = base - p_mal_batch(variants)
    scores, counts = np.zeros(n), np.zeros(n)
    for i, d in zip(starts, drops):
        scores[i:i + window] += d; counts[i:i + window] += 1
    scores = scores / np.maximum(counts, 1)
    if show:
        mx = max(np.abs(scores).max(), 1e-6)
        spans = []
        for w, s in zip(words, scores):
            a = min(abs(s) / mx, 1) * 0.75
            col = f"rgba(220,50,50,{a:.2f})" if s > 0 else f"rgba(50,110,220,{a:.2f})"
            spans.append(f"<span style='background:{col};padding:1px 2px;border-radius:3px'>{_html.escape(w)}</span>")
        display(HTML(f"<p><b>p(malicious) = {base:.3f}</b></p><div style='line-height:1.9'>{' '.join(spans)}</div>"))
        top = np.argsort(-scores)[:10]
        print("Top suspicious words:", [words[i] for i in top if scores[i] > 0])
    return words, scores

ex = test_df[test_df.label == 1].iloc[1]
_ = explain(ex["text"])

## 11. Hybrid retriever: fast BM25 + dense (RRF fusion) + optional cross-encoder re-ranker

In [ ]:
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import CrossEncoder

# ---- fast sparse BM25 over the knowledge base ----
cv = CountVectorizer(token_pattern=r"\w+", lowercase=True, dtype=np.float32)
tf = cv.fit_transform(kb["text"]).tocoo()
N_DOC = tf.shape[0]
df_t = np.bincount(tf.col, minlength=tf.shape[1])
idf = np.log((N_DOC - df_t + 0.5) / (df_t + 0.5) + 1).astype(np.float32)
dl = np.asarray(tf.tocsr().sum(1)).ravel()
avgdl = dl.mean()
BM25_K1, BM25_B = 1.5, 0.75
norm = BM25_K1 * (1 - BM25_B + BM25_B * dl[tf.row] / avgdl)
w = tf.data * (BM25_K1 + 1) / (tf.data + norm) * idf[tf.col]
BM25_W = csr_matrix((w, (tf.row, tf.col)), shape=tf.shape).T.tocsr()


def bm25_topk(qs, pool):
    Qm = cv.transform(qs)
    Qm.data[:] = 1
    S = (Qm @ BM25_W).toarray()
    return np.argsort(-S, axis=1)[:, :pool]


_reranker = None


def get_reranker():
    global _reranker
    if _reranker is None:
        _reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2",
                                 device=DEVICE, max_length=256)
    return _reranker


def dense_ids(query, k):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    return index.search(q, k)[1][0].tolist()


def rrf_fuse(d_ids, b_ids, rrf_k=60):
    fused = {}
    for ids in (d_ids, b_ids):
        for rank, i in enumerate(ids):
            fused[i] = fused.get(i, 0) + 1.0 / (rrf_k + rank + 1)
    return sorted(fused, key=fused.get, reverse=True)


def hybrid_ids(query, k=5, pool=20, rerank=False):
    d_ids = dense_ids(query, pool)
    b_ids = bm25_topk([query], pool)[0].tolist()
    cand = rrf_fuse(d_ids, b_ids)[:pool]
    if rerank:
        pairs = [(query[:600], kb["text"].iloc[i][:600]) for i in cand]
        sc = get_reranker().predict(pairs, batch_size=64, show_progress_bar=False)
        cand = [cand[j] for j in np.argsort(-sc)]
    return cand[:k]


demo_q = "Your account has been suspended, verify your password at this link"
for i in hybrid_ids(demo_q, k=3, rerank=GPU):
    print(kb["label"].iloc[i], kb["text"].iloc[i][:120])

## 12. RAG evaluation
Retrieval **label precision@k**: share of the top-k retrieved emails with the same label as the query (a proxy for relevance), plus kNN-vote accuracy.

In [ ]:
import time

EVAL_N, K, POOL = 100, 5, 20
USE_RERANK = GPU          # the cross-encoder is slow on CPU

eval_df = test_df.sample(min(EVAL_N, len(test_df)), random_state=SEED)
eval_df = eval_df.reset_index(drop=True)
queries = eval_df["text"].tolist()
y_eval = eval_df["label"].values

t0 = time.time()
q_emb = embedder.encode(
    queries, batch_size=64, normalize_embeddings=True, show_progress_bar=False
).astype("float32")
dense_top = index.search(q_emb, POOL)[1]
t_dense = time.time() - t0

t0 = time.time()
bm_top = bm25_topk(queries, POOL)
fused = []
for d, bm in zip(dense_top, bm_top):
    fused.append(rrf_fuse(d.tolist(), bm.tolist())[:POOL])
t_hybrid = t_dense + (time.time() - t0)


def evaluate(name, id_lists, secs):
    prec, vote = [], []
    for ids, yy in zip(id_lists, y_eval):
        labs = kb["label"].iloc[list(ids[:K])].values
        prec.append((labs == yy).mean())
        vote.append(int(labs.mean() > 0.5) == yy)
    return {
        "retriever": name,
        f"precision@{K}": np.mean(prec),
        "kNN-vote acc": np.mean(vote),
        "sec/query": secs / len(queries),
    }


results = [
    evaluate("Dense (FAISS)", dense_top.tolist(), t_dense),
    evaluate("Hybrid (BM25+dense)", fused, t_hybrid),
]

if USE_RERANK:
    t0 = time.time()
    rr = get_reranker()
    pairs, owner = [], []
    for qi, cands in enumerate(fused):
        for i in cands:
            pairs.append((queries[qi][:600], kb["text"].iloc[i][:600]))
            owner.append((qi, i))
    scores = rr.predict(pairs, batch_size=128, show_progress_bar=True)
    per_q = {}
    for (qi, i), s in zip(owner, scores):
        per_q.setdefault(qi, []).append((s, i))
    reranked = []
    for qi in range(len(queries)):
        ranked = sorted(per_q[qi], reverse=True)
        reranked.append([i for s, i in ranked])
    secs = t_hybrid + (time.time() - t0)
    results.append(evaluate("Hybrid + re-ranker", reranked, secs))

rag_eval = pd.DataFrame(results).round(4)
display(rag_eval)

In [ ]:
# Human-rating sheet for report quality (fill in the 1-5 columns manually, then average them)
sheet = []
for _, r in test_df.sample(10, random_state=SEED).iterrows():
    out = forensic_report(r["text"], r["sender"])
    sheet.append({"email_id": r["id"], "true_label": r["label"], "pred": out["classification"]["label"],
                  "risk": out["risk"], "report": out["report"],
                  "rating_accuracy_1to5": "", "rating_groundedness_1to5": "", "rating_usefulness_1to5": ""})
pd.DataFrame(sheet).to_csv(f"{OUT_DIR}/report_rating_sheet.csv", index=False)
print("Saved:", f"{OUT_DIR}/report_rating_sheet.csv")

## 13. Email header forensics
Works on raw `.eml` text (full headers). Checks authentication results, sender-domain mismatches, brand impersonation and the Received chain.

In [ ]:
from email import policy
from email.parser import Parser
from email.utils import parseaddr, parsedate_to_datetime

BRANDS = ["paypal", "microsoft", "amazon", "apple", "google", "netflix", "bank", "dhl", "fedex", "irs"]
IP_RE = re.compile(r"(?<![\d.])(\d{1,3}(?:\.\d{1,3}){3})(?![\d.])")

def _dom(v):
    a = parseaddr(str(v or ""))[1]
    return a.split("@")[-1].lower() if "@" in a else ""

def analyze_headers(raw):
    msg = Parser(policy=policy.default).parsestr(raw)
    flags = []
    name, addr = parseaddr(str(msg.get("From", "")))
    d_from, d_reply, d_ret = _dom(msg.get("From")), _dom(msg.get("Reply-To")), _dom(msg.get("Return-Path"))
    if d_reply and d_reply != d_from: flags.append(f"Reply-To domain ({d_reply}) differs from From domain ({d_from})")
    if d_ret and d_ret != d_from:     flags.append(f"Return-Path domain ({d_ret}) differs from From domain ({d_from})")
    for b in BRANDS:
        if b in name.lower() and b not in d_from:
            flags.append(f"Display name mentions '{b}' but sender domain is {d_from}")
    auth = str(msg.get("Authentication-Results", "")).lower()
    auth_res = {}
    for mech in ("spf", "dkim", "dmarc"):
        m = re.search(rf"{mech}=(\w+)", auth)
        auth_res[mech] = m.group(1) if m else "missing"
        if auth_res[mech] != "pass": flags.append(f"{mech.upper()} result: {auth_res[mech]}")
    hops = []
    for h in msg.get_all("Received", []):
        try: ts = parsedate_to_datetime(h.split(";")[-1].strip())
        except Exception: ts = None
        hops.append({"ips": IP_RE.findall(h), "time": ts, "text": " ".join(h.split())[:140]})
    hops.reverse()   # oldest hop first
    times = [h["time"] for h in hops if h["time"]]
    if any(b < a for a, b in zip(times, times[1:])): flags.append("Received timestamps are out of order (possible forgery)")
    if len(hops) <= 1: flags.append("Very short Received chain")
    return {"from": str(msg.get("From", "")), "reply_to": str(msg.get("Reply-To", "")),
            "return_path": str(msg.get("Return-Path", "")), "subject": str(msg.get("Subject", "")),
            "auth": auth_res, "hops": hops, "flags": flags, "header_risk": len(flags), "msg": msg}

SAMPLE_EML = "\n".join([
    "Received: from mail.evil-host.ru (unknown [185.220.101.7]) by mx.google.com with ESMTP; Mon, 26 Oct 2026 08:14:02 +0000",
    "Received: from [10.0.0.5] (localhost) by mail.evil-host.ru; Mon, 26 Oct 2026 08:13:58 +0000",
    "Authentication-Results: mx.google.com; spf=fail smtp.mailfrom=support@paypa1-secure.com; dkim=none; dmarc=fail",
    'From: "PayPal Support" <support@paypa1-secure.com>',
    "Reply-To: <collect@free-mail-box.net>",
    "Return-Path: <bounce@evil-host.ru>",
    "To: victim@gmail.com",
    "Subject: Urgent: Your account is limited",
    "Date: Mon, 26 Oct 2026 08:13:55 +0000",
    "Message-ID: <abc123@evil-host.ru>",
    "",
    "Dear customer, your account is suspended. Verify your password within 24 hours: http://185.220.101.7/paypal/login",
])

h = analyze_headers(SAMPLE_EML)
print("AUTH:", h["auth"])
print("FLAGS:"); [print(" -", f) for f in h["flags"]]
print("RECEIVED CHAIN (origin first):"); [print(f"  {i+1}. {x['time']} IPs={x['ips']}") for i, x in enumerate(h["hops"])]

In [ ]:
def investigate_eml(raw):
    h = analyze_headers(raw)
    part = h["msg"].get_body(preferencelist=("plain", "html"))
    body = part.get_content() if part else ""
    body = clean_text(body)
    rep = forensic_report(f"Subject: {h['subject']} | {body}", h["from"], k=3)
    total = rep["indicators"]["rule_risk_score"] + h["header_risk"] + (3 if rep["classification"]["p_malicious"] > 0.5 else 0)
    level = "HIGH" if total >= 7 else "MEDIUM" if total >= 4 else "LOW"
    print(f"=== COMBINED RISK: {level} (score {total}) ===")
    print("Header flags:", h["flags"] or "none")
    print("Content verdict:", rep["classification"])
    print("\n" + rep["report"])
    return h, rep

_ = investigate_eml(SAMPLE_EML)

# To analyse your own raw email, paste its full source into a string, or in Colab:
# from google.colab import files; up = files.upload(); raw = list(up.values())[0].decode(errors="ignore"); investigate_eml(raw)

In [ ]:
# Sender-domain intelligence from the dataset (if sender column is populated)
s = train_df.dropna(subset=["sender"]).copy()
s["domain"] = s["sender"].astype(str).map(lambda x: _dom(x))
s = s[s["domain"] != ""]
if len(s):
    top = s.groupby("domain")["label"].agg(["count", "mean"]).query("count >= 5").sort_values("mean", ascending=False)
    display(top.head(10).rename(columns={"mean": "malicious_ratio"}).round(3))
else:
    print("No usable sender data in this dataset sample.")

## Part C - Advanced forensic features
Runs on a **synthetic demo case** (a business-email-compromise thread with simulated headers). To use real evidence call `ingest_mbox("file.mbox")` / `ingest_eml_folder("folder")`, or upload files in the web app.

| Feature | Cells |
|---|---|
| Obfuscation decoding + OCR | C3 |
| Zero-trust PII redaction | C4 |
| Hash-cited grounding and chain of custody | C1, C5, C9, C10 |
| Hybrid header + semantic embeddings | C6, C7 |
| GraphRAG: graph, thread stitching, timeline | C8 |

### C1 - Setup and tamper-evident chain-of-custody log

In [ ]:
import os, re, io, json, hashlib, mailbox, ipaddress, difflib, unicodedata
import datetime as dt
from collections import defaultdict
from email import policy
from email.message import EmailMessage
from email.parser import BytesParser
from email.utils import parseaddr, getaddresses, parsedate_to_datetime, format_datetime
import networkx as nx

CASE_DIR = f"{OUT_DIR}/case"
os.makedirs(CASE_DIR, exist_ok=True)
TRUSTED_DOMAINS = ["acme-corp.com", "supplier-co.com"]


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_text(t):
    return sha256_bytes(t.encode("utf-8"))


# ---- tamper-evident chain-of-custody log (each entry hashes the previous one) ----
CUSTODY = []


def custody_event(action, detail):
    prev = CUSTODY[-1]["entry_hash"] if CUSTODY else "0" * 64
    ev = {"time": dt.datetime.now(dt.timezone.utc).isoformat(),
          "action": action, "detail": detail, "prev_hash": prev}
    ev["entry_hash"] = sha256_text(json.dumps(ev, sort_keys=True))
    CUSTODY.append(ev)


def verify_custody():
    prev = "0" * 64
    for e in CUSTODY:
        core = {k: e[k] for k in ("time", "action", "detail", "prev_hash")}
        ok = e["prev_hash"] == prev
        ok = ok and sha256_text(json.dumps(core, sort_keys=True)) == e["entry_hash"]
        if not ok:
            return False
        prev = e["entry_hash"]
    return True


custody_event("session_start", {"case_dir": CASE_DIR})
print("Setup OK")

### C2 - Build the synthetic demo case (.mbox)

In [ ]:
def make_png(text):
    from PIL import Image, ImageDraw, ImageFont
    img = Image.new("RGB", (1100, 120), "white")
    d = ImageDraw.Draw(img)
    try:
        font = ImageFont.load_default(size=32)
    except TypeError:
        font = ImageFont.load_default()
    d.text((20, 40), text, fill="black", font=font)
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return buf.getvalue()


def build_demo_mbox(path):
    """DEMO case: synthetic business-email-compromise thread (headers are simulated)."""
    UTC = dt.timezone.utc
    when = lambda d, h, m, s=0: dt.datetime(2026, 10, d, h, m, s, tzinfo=UTC)

    def hop(host, ip, by, t):
        return f"from {host} ([{ip}]) by {by} with ESMTPS; {format_datetime(t)}"

    def mk(mid, frm, to, subj, body, t, hops, auth, reply_to=None,
           ret=None, irt=None, refs=None, attach=None):
        m = EmailMessage()
        for h in hops:
            m["Received"] = h
        m["Authentication-Results"] = "mx.acme-corp.com; " + auth
        m["From"], m["To"], m["Subject"] = frm, to, subj
        m["Date"] = format_datetime(t)
        m["Message-ID"] = mid
        if reply_to: m["Reply-To"] = reply_to
        if ret: m["Return-Path"] = ret
        if irt: m["In-Reply-To"] = irt
        if refs: m["References"] = " ".join(refs)
        m.set_content(body)
        if attach:
            m.add_attachment(attach[1], maintype="image", subtype="png", filename=attach[0])
        return m

    OK = "spf=pass; dkim=pass; dmarc=pass"
    BAD = "spf=fail; dkim=none; dmarc=fail"
    M1, M2 = "<m1.4471@acme-corp.com>", "<m2.4471@supplier-co.com>"
    M3 = "<m3.4471@acme-corp.com>"          # this one is NOT written (deleted)
    M4, M5 = "<9f3a1c@evil-host.ru>", "<m5.4471@acme-corp.com>"
    M6, M7 = "<b21c77@evil-host.ru>", "<m7.4471@acme-corp.com>"
    D, S = "Dana Cole <dana@acme-corp.com>", "Raj Patel <raj@supplier-co.com>"
    t1, t2, t4 = when(5, 9, 12), when(5, 15, 40), when(14, 3, 21, 7)
    t5, t6, t7 = when(14, 8, 5), when(14, 9, 30), when(14, 11, 2)

    zw = "\u200b"
    body4 = ("Dear Dana,\n\nOur bank details have changed. Please w" + zw + "ire the paym\u0435nt "
             "of $48,200 for invoice 4471 today.\nNew account: 123456789012, routing "
             "021000021.\nFor the W-9 my SSN is 123-45-6789. Call me on +1 415-555-0132.\n\n"
             "Raj Patel\nSupplier Co")
    body6 = ("Dana, this is urgent. Our bank closes soon, w" + zw + "ire the funds today.\n"
             "See the attached instruction sheet.\n\nRaj")
    png = make_png("WIRE $48,200 TO ACCOUNT 123456789012 ROUTING 021000021")

    msgs = [
        mk(M1, D, "raj@supplier-co.com", "Invoice 4471 payment schedule",
           "Hi Raj,\n\nCan you confirm the payment schedule for invoice 4471?\n\nDana", t1,
           [hop("mail.acme-corp.com", "52.95.20.14", "mx.supplier-co.com", t1),
            hop("localhost", "10.0.0.5", "mail.acme-corp.com", t1)], OK),
        mk(M2, S, "dana@acme-corp.com", "Re: Invoice 4471 payment schedule",
           "Hi Dana,\n\nConfirmed. Payment is due October 30 to our usual account ending 4821.\n\nRaj",
           t2, [hop("mail.supplier-co.com", "34.102.5.12", "mx.acme-corp.com", t2),
                hop("localhost", "10.1.1.8", "mail.supplier-co.com", t2)],
           OK, irt=M1, refs=[M1]),
        mk(M4, "Raj Patel <raj@suppIier-co.com>", "dana@acme-corp.com",
           "Re: Invoice 4471 - updated bank details", body4, t4,
           [hop("mail.evil-host.ru", "185.220.101.7", "mx.acme-corp.com", t4)], BAD,
           reply_to="<collect@free-mail-box.net>", ret="<bounce@evil-host.ru>",
           irt=M3, refs=[M1, M2, M3]),
        mk(M5, D, "collect@free-mail-box.net", "Re: Invoice 4471 - updated bank details",
           "Thanks Raj. Can you confirm by phone before I send this?\n\nDana", t5,
           [hop("mail.acme-corp.com", "52.95.20.14", "mx.free-mail-box.net", t5)], OK,
           irt=M4, refs=[M1, M2, M3, M4]),
        mk(M6, "Raj Patel <raj@suppIier-co.com>", "dana@acme-corp.com",
           "Re: Invoice 4471 - updated bank details", body6, t6,
           [hop("mail.evil-host.ru", "185.220.101.7", "mx.acme-corp.com", t6)], BAD,
           reply_to="<collect@free-mail-box.net>", ret="<bounce@evil-host.ru>",
           irt=M5, refs=[M1, M2, M3, M4, M5], attach=("wire_instructions.png", png)),
        mk(M7, D, "finance@acme-corp.com", "Please process wire - invoice 4471",
           "Finance team,\n\nPlease process the wire of $48,200 to the new account "
           "123456789012 per Raj's updated instructions.\n\nDana", t7,
           [hop("mail.acme-corp.com", "52.95.20.14", "mx.acme-corp.com", t7)], OK,
           irt=M6, refs=[M4, M5, M6]),
    ]
    if os.path.exists(path):
        os.remove(path)
    mb = mailbox.mbox(path)
    mb.lock()
    for m in msgs:
        mb.add(m)
    mb.flush()
    mb.close()
    return path


DEMO_MBOX = build_demo_mbox(f"{CASE_DIR}/demo_case.mbox")
print("Demo mbox written:", DEMO_MBOX, os.path.getsize(DEMO_MBOX), "bytes")

### C3 - Obfuscation detection, normalisation and OCR

In [ ]:
# ---------- Feature 5: adversarial text and obfuscation decoding ----------
ZW_CHARS = "\u200b\u200c\u200d\u2060\ufeff\u00ad"
ZW = dict.fromkeys(map(ord, ZW_CHARS), None)
HOMOGLYPHS = {
    "\u0430": "a", "\u0435": "e", "\u043e": "o", "\u0440": "p", "\u0441": "c",
    "\u0445": "x", "\u0443": "y", "\u0456": "i", "\u0458": "j", "\u0455": "s",
    "\u0410": "A", "\u0412": "B", "\u0415": "E", "\u041a": "K", "\u041c": "M",
    "\u041d": "H", "\u041e": "O", "\u0420": "P", "\u0421": "C", "\u0422": "T",
    "\u0425": "X", "\u03bf": "o", "\u03b1": "a", "\u0391": "A", "\u039f": "O",
    "\u03c1": "p",
}


def char_script(ch):
    try:
        return unicodedata.name(ch).split()[0]
    except ValueError:
        return "UNKNOWN"


def mixed_script_tokens(text):
    out = []
    for tok in re.findall(r"\w+", text):
        scripts = {char_script(c) for c in tok if c.isalpha()}
        if "LATIN" in scripts and scripts & {"CYRILLIC", "GREEK"}:
            out.append(tok)
    return out


def analyze_obfuscation(text):
    zw = sum(1 for c in text if c in ZW_CHARS)
    return {"zero_width": zw, "mixed_script_tokens": mixed_script_tokens(text)[:10]}


def normalize_text(text):
    t = unicodedata.normalize("NFKC", text)
    t = t.translate(ZW)

    def fix(m):
        tok = m.group(0)
        if tok in mixed_script_tokens(tok):
            return "".join(HOMOGLYPHS.get(c, c) for c in tok)
        return tok

    return re.sub(r"\w+", fix, t)


def ocr_attachment(data, ctype):
    """OCR for images (and PDFs if pdf2image/poppler are installed)."""
    try:
        import pytesseract
        from PIL import Image
        if ctype.startswith("image/"):
            img = Image.open(io.BytesIO(data)).convert("L")
            return pytesseract.image_to_string(img).strip()
        if ctype == "application/pdf":
            from pdf2image import convert_from_bytes
            pages = convert_from_bytes(data, dpi=150)[:5]
            return "\n".join(pytesseract.image_to_string(p) for p in pages).strip()
    except Exception as e:
        print("  OCR skipped:", type(e).__name__)
    return ""


raw = "Please w\u200bire the paym\u0435nt now"
print(analyze_obfuscation(raw))
print(repr(normalize_text(raw)))

### C4 - PII redaction with a local vault

In [ ]:
# ---------- Feature 4: zero-trust local PII redaction ----------
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
except Exception as e:
    nlp = None
    print("spaCy model not available, using regex-only redaction:", e)


def luhn_ok(num):
    digits = [int(c) for c in num if c.isdigit()]
    if not 13 <= len(digits) <= 19:
        return False
    s = 0
    for i, d in enumerate(reversed(digits)):
        if i % 2 == 1:
            d = d * 2 - 9 if d * 2 > 9 else d * 2
        s += d
    return s % 10 == 0


class PIIVault:
    """Deterministic tokens: the same value always maps to the same token."""
    def __init__(self):
        self.fwd, self.rev, self.count = {}, {}, defaultdict(int)

    def token(self, kind, value):
        key = (kind, value.strip().lower())
        if key not in self.fwd:
            self.count[kind] += 1
            tok = f"[{kind}_{self.count[kind]}]"
            self.fwd[key] = tok
            self.rev[tok] = value
        return self.fwd[key]

    def restore(self, text):
        for tok, val in self.rev.items():
            text = text.replace(tok, val)
        return text

    def save(self, path):
        with open(path, "w") as f:
            json.dump(self.rev, f, indent=1)


VAULT = PIIVault()
REDACT_EMAILS_IN_BODY = True
PII_REGEX = [
    ("SSN", re.compile(r"\b\d{3}-\d{2}-\d{4}\b")),
    ("CARD", re.compile(r"(?<!\d)(?:\d[ -]?){13,19}(?!\d)")),
    ("PHONE", re.compile(r"(?<!\d)(?:\+?\d{1,2}[ .-]?)?\(?\d{3}\)?[ .-]\d{3}[ .-]\d{4}(?!\d)")),
    ("EMAIL", re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")),
    ("BANKNUM", re.compile(r"(?<![\d\[])\d{9,17}(?!\d)")),
]


def redact(text):
    counts = defaultdict(int)
    for kind, rx in PII_REGEX:
        if kind == "EMAIL" and not REDACT_EMAILS_IN_BODY:
            continue

        def sub(m, kind=kind):
            val = m.group(0)
            if kind == "CARD" and not luhn_ok(val):
                return val
            counts[kind] += 1
            return VAULT.token(kind, val)

        text = rx.sub(sub, text)
    if nlp is not None:
        doc = nlp(text)
        for ent in reversed(doc.ents):
            if ent.label_ == "PERSON" and len(ent.text) > 2:
                tok = VAULT.token("PERSON", ent.text)
                counts["PERSON"] += 1
                text = text[:ent.start_char] + tok + text[ent.end_char:]
    return text, dict(counts)


t, c = redact("My SSN is 123-45-6789, call +1 415-555-0132, acct 123456789012. Raj Patel")
print(t, c)

### C5 - Ingest: SHA-256 hashing, parsing, sanitising, redacting, chunking

In [ ]:
# ---------- Ingestion: hashing, parsing, sanitising, redacting, chunking ----------
CASE, REC_BY_H, EVID, RAW = [], {}, {}, {}
IP_RE2 = re.compile(r"(?<![\d.])(\d{1,3}(?:\.\d{1,3}){3})(?![\d.])")


def dom_of(v):
    a = parseaddr(str(v or ""))[1]
    return a.split("@")[-1].lower() if "@" in a else ""


def lookalike_of(domain):
    for t in TRUSTED_DOMAINS:
        if domain and domain != t:
            if difflib.SequenceMatcher(None, domain, t).ratio() >= 0.85:
                return t
    return None


def split_chunks(text, max_len=300):
    out = []
    for para in re.split(r"\n\s*\n", text):
        para = " ".join(para.split())
        if not para:
            continue
        if len(para) <= max_len:
            out.append(para)
        else:
            out += [s for s in re.split(r"(?<=[.!?])\s+", para) if s]
    return out


def register(full_hash, kind, text, email_h12):
    h12 = full_hash[:12]
    EVID[h12] = {"full": full_hash, "kind": kind, "text": text, "email": email_h12}
    return h12


def clean_section(text):
    obf = analyze_obfuscation(text)
    red, pii = redact(normalize_text(text))
    return red, obf, pii


def parse_raw(raw, source):
    msg = BytesParser(policy=policy.default).parsebytes(raw)
    esha = sha256_bytes(raw)
    h12 = esha[:12]
    RAW[h12] = raw
    part = msg.get_body(preferencelist=("plain", "html"))
    body = part.get_content() if part else ""
    if part is not None and part.get_content_type() == "text/html":
        body = re.sub(r"<[^>]+>", " ", body)
    subj_raw = str(msg.get("Subject", ""))

    body_red, obf, pii = clean_section(body)
    subj_red, obf_s, _ = clean_section(subj_raw)
    obf["zero_width"] += obf_s["zero_width"]
    obf["mixed_script_tokens"] += obf_s["mixed_script_tokens"]

    chunks, atts = [], []
    for c in split_chunks(body_red):
        full = sha256_text(f"{esha}|body|{c}")
        chunks.append({"h12": register(full, "body_chunk", c, h12), "kind": "body", "text": c})
    for p in msg.iter_attachments():
        data = p.get_payload(decode=True) or b""
        name = p.get_filename() or "attachment"
        txt = ocr_attachment(data, p.get_content_type())
        a_sha = sha256_bytes(data)
        if txt:
            t_red, o, pz = clean_section(txt)
            t_red = " ".join(t_red.split())
            full = sha256_text(f"{esha}|att|{a_sha}|{t_red}")
            ah = register(full, "attachment_ocr", t_red, h12)
            atts.append({"name": name, "sha256": a_sha, "h12": ah, "text": t_red})
            chunks.append({"h12": ah, "kind": "attachment", "text": t_red})
            obf["zero_width"] += o["zero_width"]
            obf["mixed_script_tokens"] += o["mixed_script_tokens"]
            for k, v in pz.items():
                pii[k] = pii.get(k, 0) + v

    auth_hdr = " ".join(str(x) for x in msg.get_all("Authentication-Results", [])).lower()
    auth = {}
    for k in ("spf", "dkim", "dmarc"):
        m = re.search(rf"{k}=(\w+)", auth_hdr)
        auth[k] = m.group(1) if m else "missing"

    hops, ips = [], []
    for h in msg.get_all("Received", []):
        h = " ".join(str(h).split())
        try:
            ts = parsedate_to_datetime(h.split(";")[-1].strip())
        except Exception:
            ts = None
        found = IP_RE2.findall(h)
        hops.append({"ips": found, "time": ts})
        ips += [i for i in found if i not in ips]
    hops.reverse()
    pub = []
    for i in ips:
        try:
            if ipaddress.ip_address(i).is_global:
                pub.append(i)
        except ValueError:
            pass

    try:
        date = parsedate_to_datetime(str(msg["Date"]))
    except Exception:
        date = dt.datetime(1970, 1, 1, tzinfo=dt.timezone.utc)
    mid = str(msg.get("Message-ID", "")).strip()
    irt = str(msg.get("In-Reply-To", "")).strip()
    refs = str(msg.get("References", "")).split()
    d_from = dom_of(msg.get("From"))
    d_reply, d_ret = dom_of(msg.get("Reply-To")), dom_of(msg.get("Return-Path"))
    d_mid = mid.strip("<>").split("@")[-1].lower() if "@" in mid else ""
    to = [a for _, a in getaddresses([str(x) for x in msg.get_all("To", [])]) if a]
    bits = {"reply": int(bool(d_reply) and d_reply != d_from),
            "ret": int(bool(d_ret) and d_ret != d_from),
            "msgid": int(bool(d_mid) and d_mid != d_from),
            "lookalike": int(lookalike_of(d_from) is not None)}
    flags = []
    for k in ("spf", "dkim", "dmarc"):
        if auth[k] not in ("pass", "missing"):
            flags.append(f"{k.upper()} {auth[k]}")
    if bits["reply"]:
        flags.append(f"Reply-To domain {d_reply} differs from From domain {d_from}")
    if bits["ret"]:
        flags.append(f"Return-Path domain {d_ret} differs from From domain {d_from}")
    if bits["msgid"]:
        flags.append(f"Message-ID domain {d_mid} differs from From domain {d_from}")
    if bits["lookalike"]:
        flags.append(f"Sender domain {d_from} imitates trusted domain {lookalike_of(d_from)}")
    if obf["zero_width"]:
        flags.append(f"{obf['zero_width']} zero-width characters hidden in text")
    if obf["mixed_script_tokens"]:
        flags.append("Mixed-script (homoglyph) words detected")

    ind = forensic_indicators(body_red)
    from_addr = parseaddr(str(msg.get("From", "")))[1].lower()
    rec = dict(h12=h12, sha=esha, source=source, message_id=mid, in_reply_to=irt, refs=refs,
               from_addr=from_addr, to=[t.lower() for t in to], subject=subj_red,
               reply_to=d_reply, return_path=d_ret, date=date,
               date_iso=date.isoformat(), auth=auth, hops=hops, ips=ips,
               n_hops=len(hops), n_public_ips=len(pub), obf=obf, pii=pii,
               n_urls=ind["num_urls"], ip_url=int(ind["ip_based_url"]),
               urgency=len(ind["urgency_terms"]), flags=flags, bits=bits,
               chunks=chunks, att=atts)
    rec["meta_text"] = (f"Message-ID {mid}; From {from_addr}; To {', '.join(rec['to'])}; "
                        f"Reply-To {d_reply}; Return-Path {d_ret}; Date {rec['date_iso']}; "
                        f"Subject {subj_red}; SPF {auth['spf']}; DKIM {auth['dkim']}; "
                        f"DMARC {auth['dmarc']}; Relay IPs {', '.join(ips)}; Hops {len(hops)}; "
                        f"In-Reply-To {irt}; References {' '.join(refs)}; "
                        f"Flags: {'; '.join(flags) if flags else 'none'}")
    register(esha, "email_artifact", rec["meta_text"], h12)
    rec["text_for_embed"] = f"{subj_red}. " + " ".join(c["text"] for c in chunks)
    return rec


def ingest_mbox(path):
    mb = mailbox.mbox(path)
    n = 0
    for key in mb.keys():
        raw = mb.get_bytes(key)
        rec = parse_raw(raw, os.path.basename(path))
        CASE.append(rec)
        REC_BY_H[rec["h12"]] = rec
        custody_event("ingest_email", {"sha256": rec["sha"], "source": rec["source"]})
        n += 1
    CASE.sort(key=lambda r: r["date"])
    return n


def ingest_eml_folder(folder):
    n = 0
    for f in sorted(os.listdir(folder)):
        if f.lower().endswith(".eml"):
            raw = open(os.path.join(folder, f), "rb").read()
            rec = parse_raw(raw, f)
            CASE.append(rec)
            REC_BY_H[rec["h12"]] = rec
            custody_event("ingest_email", {"sha256": rec["sha"], "source": f})
            n += 1
    CASE.sort(key=lambda r: r["date"])
    return n


src_hash = sha256_bytes(open(DEMO_MBOX, "rb").read())
custody_event("source_file", {"file": "demo_case.mbox", "sha256": src_hash})
print("Ingested emails:", ingest_mbox(DEMO_MBOX))
for r in CASE:
    print(r["h12"], r["date_iso"][:16], r["from_addr"], "|", r["subject"][:35],
          "| flags:", len(r["flags"]))
VAULT.save(f"{CASE_DIR}/pii_vault.json")

### C6 - Structure encoder (MLP) for header features (pre-trained on SIMULATED header profiles)

In [ ]:
# ---------- Feature 1a: structure encoder (MLP) for header/metadata features ----------
import torch.nn as nn

AUTH_KEYS = ["spf", "dkim", "dmarc"]


def auth_onehot(v):
    bad = ("fail", "softfail", "permerror", "none", "temperror")
    return [v == "pass", v in bad, (v not in bad) and v not in ("pass", "missing"), v == "missing"]


def struct_features(r):
    f = []
    for k in AUTH_KEYS:
        f += auth_onehot(r["auth"].get(k, "missing"))
    f += [np.log1p(r["n_hops"]) / 2, np.log1p(r["n_public_ips"]) / 2]
    b = r["bits"]
    f += [b["reply"], b["ret"], b["msgid"], b["lookalike"]]
    f += [np.log1p(r["obf"]["zero_width"]) / 2, np.log1p(len(r["obf"]["mixed_script_tokens"])) / 2]
    f += [np.log1p(r["n_urls"]) / 2, r["ip_url"], min(r["urgency"], 3) / 3]
    no_hdr = all(r["auth"].get(k, "missing") == "missing" for k in AUTH_KEYS) and r["n_hops"] == 0
    f += [int(no_hdr)]
    return np.array(f, dtype="float32")


def profile_rec(**kw):
    """Build a pseudo-record from a few header facts, e.g. profile_rec(dkim='fail')."""
    r = {"auth": {k: kw.get(k, "missing") for k in AUTH_KEYS},
         "n_hops": kw.get("n_hops", 1), "n_public_ips": kw.get("n_public_ips", 1),
         "bits": {"reply": kw.get("reply", 0), "ret": kw.get("ret", 0),
                  "msgid": kw.get("msgid", 0), "lookalike": kw.get("lookalike", 0)},
         "obf": {"zero_width": kw.get("zero_width", 0),
                 "mixed_script_tokens": [1] * kw.get("mixed", 0)},
         "n_urls": kw.get("n_urls", 0), "ip_url": kw.get("ip_url", 0),
         "urgency": kw.get("urgency", 0)}
    return r


STRUCT_DIM = len(struct_features(profile_rec()))
STRUCT_EMB = 32


class StructEncoder(nn.Module):
    def __init__(self, d_in=STRUCT_DIM, d_emb=STRUCT_EMB):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d_in, 64), nn.ReLU(), nn.Linear(64, d_emb))
        self.head = nn.Sequential(nn.ReLU(), nn.Linear(d_emb, 2))

    def forward(self, x):
        z = self.enc(x)
        return self.head(z), z


# --- SIMULATED header profiles for pre-training (the Kaggle dataset has no raw headers) ---
def sim_record(text, label, rng):
    ind = forensic_indicators(text)
    p_bad = 0.65 if label == 1 else 0.05
    pick = lambda: str(rng.choice(["fail", "softfail", "none"])) if rng.random() < p_bad else "pass"
    p = lambda a, b: int(rng.random() < (a if label == 1 else b))
    r = profile_rec(spf=pick(), dkim=pick(), dmarc=pick(),
                    n_hops=int(rng.integers(1, 7)), n_public_ips=int(rng.integers(0, 4)),
                    reply=p(0.5, 0.03), ret=p(0.4, 0.03), msgid=p(0.4, 0.05),
                    lookalike=p(0.3, 0.01), zero_width=p(0.1, 0.005), mixed=p(0.1, 0.005),
                    n_urls=ind["num_urls"], ip_url=int(ind["ip_based_url"]),
                    urgency=len(ind["urgency_terms"]))
    return r


rng = np.random.default_rng(SEED)
samp = train_df.sample(min(6000, len(train_df)), random_state=SEED)
X = np.stack([struct_features(sim_record(t, y, rng))
              for t, y in zip(samp["text"], samp["label"])])
y_s = samp["label"].values
cut = int(0.8 * len(X))
Xs_tr, Xs_va = torch.tensor(X[:cut]), torch.tensor(X[cut:])
ytr, yva = torch.tensor(y_s[:cut]), torch.tensor(y_s[cut:])

struct_model = StructEncoder().to(DEVICE)
opt = torch.optim.Adam(struct_model.parameters(), lr=2e-3)
lossf = nn.CrossEntropyLoss()
for epoch in range(40):
    struct_model.train()
    perm = torch.randperm(len(Xs_tr))
    for i in range(0, len(perm), 256):
        idx = perm[i:i + 256]
        logits, _ = struct_model(Xs_tr[idx].to(DEVICE))
        loss = lossf(logits, ytr[idx].to(DEVICE))
        opt.zero_grad()
        loss.backward()
        opt.step()
struct_model.eval()
with torch.no_grad():
    acc = (struct_model(Xs_va.to(DEVICE))[0].argmax(-1).cpu() == yva).float().mean().item()
print(f"Structure encoder holdout accuracy (simulated headers): {acc:.3f}")
torch.save(struct_model.state_dict(), f"{OUT_DIR}/struct_encoder.pt")


@torch.no_grad()
def struct_embed(recs):
    x = torch.tensor(np.stack([struct_features(r) for r in recs])).to(DEVICE)
    z = struct_model(x)[1].cpu().numpy()
    return (z / (np.linalg.norm(z, axis=1, keepdims=True) + 1e-8)).astype("float32")


print("Structure vector dim:", STRUCT_EMB)

### C7 - Hybrid header + semantic vectors and retrieval demo

In [ ]:
# ---------- Feature 1b: hybrid header + semantic vectors ----------
# Self-checking: if CASE is empty it loads the demo case automatically.
need = ["embedder", "struct_embed", "profile_rec", "CASE", "ingest_mbox", "DEMO_MBOX"]
missing = [n for n in need if n not in globals()]
if missing:
    raise RuntimeError(f"Run the earlier cells first (C1-C6). Missing: {missing}")

if len(CASE) == 0:
    print("CASE is empty - loading the demo case now...")
    try:
        print("Ingested emails:", ingest_mbox(DEMO_MBOX))
    except NameError as e:
        raise RuntimeError(f"{e}. Run the C4 (PII redaction) cell first, then rerun this cell.")
assert len(CASE) > 0, "No emails loaded - check the C5 ingest cell."

ALPHA = 0.8   # weight of the structure vector (0 = text-only)


def build_case_index(alpha=ALPHA):
    global CASE_TEXT, CASE_STRUCT, CASE_HYB
    CASE_TEXT = embedder.encode(
        [r["text_for_embed"] for r in CASE],
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")
    CASE_STRUCT = struct_embed(CASE)
    # one concatenated vector per email: [text embedding | alpha * structure embedding]
    CASE_HYB = np.hstack([CASE_TEXT, alpha * CASE_STRUCT])
    return CASE_HYB.shape


def hybrid_search(query, profile=None, k=3, alpha=ALPHA):
    qt = embedder.encode([query], normalize_embeddings=True).astype("float32")[0]
    if profile is None:
        qs = np.zeros(CASE_STRUCT.shape[1], dtype="float32")
    else:
        qs = struct_embed([profile_rec(**profile)])[0]
    hq = np.hstack([qt, alpha * qs])
    scores = np.hstack([CASE_TEXT, alpha * CASE_STRUCT]) @ hq
    top = np.argsort(-scores)[:k]
    return [
        (CASE[i], float(scores[i]), float(CASE_TEXT[i] @ qt), float(CASE_STRUCT[i] @ qs))
        for i in top
    ]


def show_hits(title, res):
    print("\n" + title)
    for r, tot, st, ss in res:
        a = r["auth"]
        print(
            f"  {tot:.3f} (text {st:.2f}, struct {ss:.2f}) [{r['h12']}] "
            f"{r['date_iso'][:10]} {r['from_addr']:<24} "
            f"spf={a['spf']} dkim={a['dkim']}"
        )


print("Emails in case:", len(CASE))
print("Hybrid vector shape:", build_case_index())

QUERY = "please wire the payment to the new bank account"
BAD_PROFILE = dict(spf="fail", dkim="none", dmarc="fail", reply=1, msgid=1, lookalike=1)
GOOD_PROFILE = dict(spf="pass", dkim="pass", dmarc="pass")

show_hits("Text-only retrieval:", hybrid_search(QUERY, None, alpha=0.0))
show_hits("Hybrid, query profile = failing authentication:",
          hybrid_search(QUERY, BAD_PROFILE))
show_hits("Hybrid, query profile = passing authentication:",
          hybrid_search(QUERY, GOOD_PROFILE))

### C8 - Entity graph, thread stitching, timeline, compromise start

In [ ]:
# ---------- Feature 3: GraphRAG - entity graph, thread stitching, timeline ----------
def missing_windows(recs):
    """Message-IDs referenced by emails but absent from the evidence (possible deletions)."""
    by_mid = {r["message_id"]: r for r in recs}
    out = {}
    for r in recs:
        refs = list(r["refs"])
        if r["in_reply_to"] and r["in_reply_to"] not in refs:
            refs.append(r["in_reply_to"])
        for i, mid in enumerate(refs):
            if mid in by_mid:
                continue
            w = out.setdefault(mid, {"lo": None, "hi": None, "children": []})
            lo = [by_mid[x]["date"] for x in refs[:i] if x in by_mid]
            if lo:
                w["lo"] = max(lo + ([w["lo"]] if w["lo"] else []))
            w["hi"] = r["date"] if w["hi"] is None else min(w["hi"], r["date"])
            w["children"].append(r["h12"])
    return out


def build_graph(recs):
    G = nx.MultiDiGraph()
    by_mid = {r["message_id"]: r for r in recs}
    for r in recs:
        e = f"email:{r['h12']}"
        G.add_node(e, kind="Email", date=r["date_iso"], flags=len(r["flags"]))
        a = f"addr:{r['from_addr']}"
        G.add_node(a, kind="Address")
        G.add_edge(a, e, rel="SENT")
        d = f"dom:{dom_of(r['from_addr'])}"
        G.add_node(d, kind="Domain")
        G.add_edge(a, d, rel="HAS_DOMAIN")
        t = lookalike_of(dom_of(r["from_addr"]))
        if t:
            G.add_node(f"dom:{t}", kind="Domain")
            G.add_edge(d, f"dom:{t}", rel="LOOKALIKE_OF")
        for to in r["to"]:
            G.add_node(f"addr:{to}", kind="Address")
            G.add_edge(e, f"addr:{to}", rel="SENT_TO")
        for ip in r["ips"]:
            G.add_node(f"ip:{ip}", kind="IP")
            G.add_edge(e, f"ip:{ip}", rel="ROUTED_THROUGH")
    for mid in missing_windows(recs):
        G.add_node(f"missing:{mid}", kind="MissingEmail")
    for r in recs:
        e = f"email:{r['h12']}"
        for ref in r["refs"]:
            tgt = f"email:{by_mid[ref]['h12']}" if ref in by_mid else f"missing:{ref}"
            if ref != r["in_reply_to"]:
                G.add_edge(e, tgt, rel="REFERENCES")
        if r["in_reply_to"]:
            ref = r["in_reply_to"]
            tgt = f"email:{by_mid[ref]['h12']}" if ref in by_mid else f"missing:{ref}"
            G.add_edge(e, tgt, rel="REPLIED_TO")
    return G


def thread_timeline(G, recs):
    wins = missing_windows(recs)
    H = nx.Graph()
    for u, v, d in G.edges(data=True):
        if d["rel"] in ("REPLIED_TO", "REFERENCES"):
            H.add_edge(u, v)
    out = []
    for comp in nx.connected_components(H):
        events = []
        for n in comp:
            if n.startswith("email:"):
                r = REC_BY_H[n.split(":", 1)[1]]
                events.append((r["date"], "email", r))
            else:
                mid = n.split(":", 1)[1]
                w = wins[mid]
                when = (w["lo"] + dt.timedelta(seconds=1)) if w["lo"] else w["hi"]
                events.append((when - dt.timedelta(seconds=1), "missing", (mid, w)))
        events.sort(key=lambda x: x[0])
        out.append(events)
    return out


def print_timeline(timeline):
    for k, events in enumerate(timeline, 1):
        print(f"\n=== Thread {k} ({len(events)} items) ===")
        for when, kind, obj in events:
            if kind == "email":
                r = obj
                mark = "!! " if r["flags"] else "   "
                print(f"{mark}{r['date_iso'][:16]} [{r['h12']}] {r['from_addr']} -> "
                      f"{', '.join(r['to'])}")
                for f in r["flags"]:
                    print(f"        - {f}")
            else:
                mid, w = obj
                lo = w["lo"].isoformat()[:16] if w["lo"] else "?"
                hi = w["hi"].isoformat()[:16] if w["hi"] else "?"
                print(f"?? MISSING {mid}  (expected between {lo} and {hi}; "
                      f"referenced by {w['children']})")


def find_compromise_start(G, recs):
    bad = sorted([r for r in recs if r["flags"]], key=lambda r: r["date"])
    if not bad:
        return None
    first = bad[0]
    prior_ips = {ip for r in recs if r["date"] < first["date"] for ip in r["ips"]}
    R = nx.DiGraph([(u, v) for u, v, d in G.edges(data=True) if d["rel"] == "REPLIED_TO"])
    node = f"email:{first['h12']}"
    down = nx.ancestors(R, node) if node in R else set()
    follow = sorted([REC_BY_H[n.split(":", 1)[1]] for n in down if n.startswith("email:")],
                    key=lambda r: r["date"])
    return {"first": first, "new_ips": [i for i in first["ips"] if i not in prior_ips],
            "downstream": follow}


def graph_context(G, seed_h12s, radius=3):
    T = nx.Graph()
    for u, v, d in G.edges(data=True):
        if d["rel"] in ("REPLIED_TO", "REFERENCES", "ROUTED_THROUGH"):
            T.add_edge(u, v)
    nodes = set()
    for h in seed_h12s:
        n = f"email:{h}"
        if n in T:
            nodes |= set(nx.ego_graph(T, n, radius=radius).nodes)
        else:
            nodes.add(n)
    recs = [REC_BY_H[n.split(":", 1)[1]] for n in nodes if n.startswith("email:")]
    return sorted(recs, key=lambda r: r["date"])


def draw_graph(G, show=True):
    import matplotlib.pyplot as plt
    col = {"Email": "#4C78A8", "Address": "#F58518", "IP": "#E45756",
           "Domain": "#72B7B2", "MissingEmail": "#999999"}
    pos = nx.spring_layout(nx.Graph(G), seed=SEED, k=0.9)
    plt.figure(figsize=(13, 8))
    nx.draw_networkx_nodes(G, pos, node_color=[col[G.nodes[n]["kind"]] for n in G],
                           node_size=520)
    nx.draw_networkx_edges(G, pos, alpha=0.35, arrows=True)
    nx.draw_networkx_labels(G, pos, labels={n: n.split(":", 1)[1][:20] for n in G},
                            font_size=7)
    plt.title("Entity graph (blue=email, orange=address, red=IP, teal=domain, grey=missing)")
    plt.axis("off")
    fig = plt.gcf()
    if show:
        plt.show()
    return fig


def to_cypher(G):
    """Optional: statements to load the same graph into Neo4j."""
    lines = [f"MERGE (:{d['kind']} {{id: {json.dumps(n)}}});" for n, d in G.nodes(data=True)]
    for u, v, d in G.edges(data=True):
        lines.append(f"MATCH (a {{id:{json.dumps(u)}}}),(b {{id:{json.dumps(v)}}}) "
                     f"MERGE (a)-[:{d['rel']}]->(b);")
    return "\n".join(lines)


GRAPH = build_graph(CASE)
print("Graph:", GRAPH.number_of_nodes(), "nodes,", GRAPH.number_of_edges(), "edges")
print_timeline(thread_timeline(GRAPH, CASE))
cs = find_compromise_start(GRAPH, CASE)
print("\nEarliest anomalous email:", cs["first"]["h12"], cs["first"]["date_iso"])
print("New relay IPs first seen there:", cs["new_ips"])
print("Downstream emails:", [r["h12"] for r in cs["downstream"]])
_ = draw_graph(GRAPH)

### C9 - Hash-cited, guard-railed investigation report

In [ ]:
# ---------- Feature 2: evidence grounding with hash citations ----------
VAL_RE = re.compile(r"[\w.+-]+@[\w.-]+\.\w+|\b\d{1,3}(?:\.\d{1,3}){3}\b|\b\d{4}-\d{2}-\d{2}\b"
                    r"|\$[\d,]+(?:\.\d+)?|\b\d{4,}\b|\[[A-Z_]+_\d+\]|<[^>\s]+>")


def evidence_pack(recs):
    lines = []
    for r in recs:
        lines.append(f"[{r['h12']}] EMAIL-METADATA {r['meta_text']}")
        for c in r["chunks"]:
            lines.append(f"[{c['h12']}] ({c['kind']} of email {r['h12']}) {c['text']}")
    return "\n".join(lines)


def verify_statement(stmt, cites):
    if not cites:
        return False, "no citation"
    for c in cites:
        if c not in EVID:
            return False, f"unknown hash {c}"
    corpus = " ".join(EVID[c]["text"] for c in cites).lower()
    missing = [v for v in VAL_RE.findall(stmt) if v.lower() not in corpus]
    if missing:
        return False, f"values not found in cited evidence: {missing}"
    return True, "ok"


def ground_check(findings):
    ok, bad = [], []
    for stmt, cites in findings:
        passed, why = verify_statement(stmt, cites)
        (ok if passed else bad).append((stmt, cites, why))
    return ok, bad


def deterministic_findings(recs):
    """Template findings built directly from stored evidence - cannot hallucinate."""
    out = []
    for r in recs:
        if r["flags"]:
            s = (f"On {r['date_iso']}, message {r['message_id']} from {r['from_addr']} to "
                 f"{', '.join(r['to'])} showed: {'; '.join(r['flags'])}.")
            out.append((s, [r["h12"]]))
        for a in r["att"]:
            s = (f"Attachment {a['name']} in message {r['message_id']} contains the text: "
                 f"{a['text'][:100]}")
            out.append((s, [a["h12"], r["h12"]]))
    wins = missing_windows(recs)
    by_h = {r["h12"]: r for r in recs}
    for mid, w in wins.items():
        ch = by_h.get(w["children"][0])
        if ch:
            s = (f"Message {mid} is referenced by {ch['message_id']} but is absent from the "
                 f"evidence, so it may have been deleted.")
            out.append((s, [ch["h12"]]))
    return out


LLM_SYSTEM = ("You are a digital-forensics analyst. Use ONLY the evidence lines provided. "
              "Return ONLY JSON: {\"findings\":[{\"statement\":\"...\",\"citations\":[\"hash12\"]}]}. "
              "Every statement must cite one or more bracketed 12-character hashes from the "
              "evidence. Copy dates, addresses, IPs and amounts exactly. Do not speculate.")


def llm_findings(question, pack):
    g = globals()
    if g.get("llm") is None:
        return []
    msgs = [{"role": "system", "content": LLM_SYSTEM},
            {"role": "user", "content": f"EVIDENCE:\n{pack}\n\nQUESTION: {question}"}]
    text = g["llm_tok"].apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = g["llm_tok"](text, return_tensors="pt").to(g["llm"].device)
    with torch.no_grad():
        out = g["llm"].generate(**inputs, max_new_tokens=700, do_sample=False)
    raw = g["llm_tok"].decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        data = json.loads(raw[raw.index("{"): raw.rindex("}") + 1])
        return [(f.get("statement", ""), [str(c).strip("[] ") for c in f.get("citations", [])])
                for f in data.get("findings", [])]
    except Exception:
        return []


def verify_integrity():
    """Recompute every hash from stored raw bytes / text. Returns list of problems."""
    problems = []
    for h12, raw in RAW.items():
        if sha256_bytes(raw) != REC_BY_H[h12]["sha"]:
            problems.append(f"raw email {h12} was modified")
    for r in CASE:
        for c in r["chunks"]:
            e = EVID[c["h12"]]
            if e["kind"] == "body_chunk":
                if sha256_text(f"{r['sha']}|body|{c['text']}")[:12] != c["h12"]:
                    problems.append(f"chunk {c['h12']} was modified")
    if not verify_custody():
        problems.append("custody log broken")
    return problems


def investigate(question, k=2, use_llm=True):
    seeds = [r["h12"] for r, *_ in hybrid_search(question, None, k=k)]
    ctx = graph_context(GRAPH, seeds)
    pack = evidence_pack(ctx)
    findings = llm_findings(question, pack) if use_llm else []
    source = "LLM (verified)"
    ok, bad = ground_check(findings) if findings else ([], [])
    if not ok:
        source = "deterministic template (LLM unavailable or nothing passed verification)"
        ok, extra = ground_check(deterministic_findings(ctx))
        bad = bad + extra
    print(f"QUESTION: {question}")
    print(f"Seed emails: {seeds} -> graph context: {[r['h12'] for r in ctx]}")
    print(f"Report source: {source}\n")
    for stmt, cites, _ in ok:
        print(f"- {stmt} " + " ".join(f"[{c}]" for c in cites))
    if bad:
        print(f"\n{len(bad)} statement(s) REJECTED by the guardrail:")
        for stmt, cites, why in bad:
            print(f"  x {stmt[:90]}... -> {why}")
    probs = verify_integrity()
    print("\nIntegrity check:", "PASS" if not probs else probs)
    print("Evidence appendix (full SHA-256):")
    used = sorted({c for _, cs, _ in ok for c in cs})
    for c in used:
        print(f"  [{c}] {EVID[c]['kind']:<15} {EVID[c]['full']}")
    return ok, bad


_ = investigate("How did the compromise start and which emails were affected?")

### C10 - Save the case report and test tamper detection

In [ ]:
# ---------- Save the case report, then try tamper detection ----------
report = {
    "case_dir": CASE_DIR,
    "emails": [{"h12": r["h12"], "sha256": r["sha"], "date": r["date_iso"],
                "from": r["from_addr"], "to": r["to"], "flags": r["flags"]} for r in CASE],
    "custody_log": CUSTODY,
    "custody_valid": verify_custody(),
}
with open(f"{CASE_DIR}/case_report.json", "w") as f:
    json.dump(report, f, indent=1)
with open(f"{CASE_DIR}/graph.cypher", "w") as f:
    f.write(to_cypher(GRAPH))
print("Saved case_report.json, graph.cypher, pii_vault.json in", CASE_DIR)

# Tamper test: change one byte of a stored email and re-verify
h = CASE[0]["h12"]
original = RAW[h]
RAW[h] = original[:-1] + b"X"
print("After tampering:", verify_integrity())
RAW[h] = original
print("After restoring:", verify_integrity() or "PASS")

## Part D - Secure professional web app (Gradio) with e-mail login
Users create an account with an e-mail address, verify it with a 6-digit code sent to their inbox, then sign in. Only after signing in is the application shown, and every server-side action re-checks the session.

### One-time e-mail setup (Gmail)
1. Google Account -> Security -> turn on **2-Step Verification**, then **App passwords** -> create one named `Email Forensics` -> copy the 16-character password.
2. In Colab click the **key icon (Secrets)** in the left sidebar -> *Add new secret*: `SMTP_USER` = your Gmail address, `SMTP_PASSWORD` = the app password. Switch **Notebook access** ON for both.
3. Run the next cells. The setup cell should print `e-mail delivery configured: True`.

Never paste the app password into a cell. For testing without e-mail, set `DEV_SHOW_OTP = True` in the auth cell (codes then appear on screen - testing only). The first account created becomes the administrator.

In [ ]:
# ======================= Secure login: accounts, email verification, sessions, audit log =======================
import sqlite3, secrets, hashlib, hmac, re, time, smtplib, ssl, contextlib
import datetime as _dt
from email.message import EmailMessage as _Mail

# ---- settings -------------------------------------------------------------------------------------------
AUTH_ON_DRIVE = False                 # True = keep the user database on Google Drive (survives resets)
REQUIRE_EMAIL_VERIFICATION = True     # users must enter the code emailed to them
DEV_SHOW_OTP = False                  # True = show codes on screen if e-mail is not configured (testing ONLY)
ALLOWED_EMAIL_DOMAINS = []            # e.g. ["gmail.com", "mycollege.edu"]; empty = any domain
ADMIN_EMAILS = []                     # these addresses become admins; the first account is admin if empty
SESSION_HOURS, OTP_MINUTES, MAX_OTP_TRIES = 8, 10, 5
MAX_FAILED_LOGINS, LOCK_MINUTES = 5, 5
APP_NAME = "Email Forensics Workbench"

_drive_dir = "/content/drive/MyDrive/email_forensics_project/auth"
AUTH_DIR = _drive_dir if (AUTH_ON_DRIVE and os.path.isdir("/content/drive/MyDrive")) else f"{OUT_DIR}/auth"
os.makedirs(AUTH_DIR, exist_ok=True)
AUTH_DB = f"{AUTH_DIR}/users.db"
SESSIONS = {}


@contextlib.contextmanager
def _db():
    con = sqlite3.connect(AUTH_DB, timeout=10)
    con.row_factory = sqlite3.Row
    try:
        yield con
        con.commit()
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()


def _init_db():
    with _db() as con:
        con.executescript("""
        CREATE TABLE IF NOT EXISTS users(
            email TEXT PRIMARY KEY, name TEXT, salt TEXT, pw_hash TEXT, role TEXT,
            status TEXT, created REAL, failed INTEGER DEFAULT 0, locked_until REAL DEFAULT 0, last_login REAL);
        CREATE TABLE IF NOT EXISTS otp(
            email TEXT, purpose TEXT, salt TEXT, code_hash TEXT, expires REAL, tries INTEGER DEFAULT 0,
            PRIMARY KEY(email, purpose));
        CREATE TABLE IF NOT EXISTS audit(
            id INTEGER PRIMARY KEY AUTOINCREMENT, ts REAL, email TEXT, action TEXT, detail TEXT);
        """)


_init_db()


def audit(email, action, detail=""):
    with _db() as con:
        con.execute("INSERT INTO audit(ts,email,action,detail) VALUES(?,?,?,?)",
                    (time.time(), email or "-", action, str(detail)[:200]))


def _hash_pw(password, salt):
    return hashlib.scrypt(password.encode(), salt=bytes.fromhex(salt), n=2 ** 14, r=8, p=1, dklen=32).hex()


def _new_salt():
    return secrets.token_hex(16)


EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")


def norm_email(e):
    return (e or "").strip().lower()


def check_password_policy(pw):
    if len(pw or "") < 8:
        return "Password must be at least 8 characters."
    if not re.search(r"[A-Za-z]", pw) or not re.search(r"\d", pw):
        return "Password must contain letters and digits."
    return None


# ---- e-mail delivery (Gmail SMTP; credentials come from Colab Secrets, never from the notebook) ---------
def _smtp_creds():
    user = os.environ.get("SMTP_USER")
    pw = os.environ.get("SMTP_PASSWORD")
    if not (user and pw):
        try:
            from google.colab import userdata
            user = user or userdata.get("SMTP_USER")
            pw = pw or userdata.get("SMTP_PASSWORD")
        except Exception:
            pass
    return user, pw


def email_configured():
    u, p = _smtp_creds()
    return bool(u and p)


def send_email(to, subject, body):
    user, pw = _smtp_creds()
    if not (user and pw):
        return False, "E-mail delivery is not configured (add SMTP_USER and SMTP_PASSWORD in Colab Secrets)."
    try:
        m = _Mail()
        m["From"], m["To"], m["Subject"] = f"{APP_NAME} <{user}>", to, subject
        m.set_content(body)
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=20) as s:
            s.login(user, pw.replace(" ", ""))
            s.send_message(m)
        return True, "sent"
    except Exception as e:
        return False, f"Could not send e-mail: {type(e).__name__}"


def _issue_otp(email, purpose):
    code = f"{secrets.randbelow(10 ** 6):06d}"
    salt = _new_salt()
    with _db() as con:
        con.execute("REPLACE INTO otp(email,purpose,salt,code_hash,expires,tries) VALUES(?,?,?,?,?,0)",
                    (email, purpose, salt, hashlib.sha256((salt + code).encode()).hexdigest(),
                     time.time() + OTP_MINUTES * 60))
    subject = f"{APP_NAME}: your verification code" if purpose == "register" else f"{APP_NAME}: password reset code"
    body = (f"Your {APP_NAME} code is {code}\n\nIt expires in {OTP_MINUTES} minutes. "
            "If you did not request it, ignore this e-mail.")
    ok, msg = send_email(email, subject, body)
    if ok:
        return True, f"A 6-digit code was sent to {email}.", None
    if DEV_SHOW_OTP:
        return True, f"DEV MODE (e-mail not sent): your code is {code}", code
    return False, msg, None


def verify_otp(email, code, purpose):
    email, code = norm_email(email), (code or "").strip()
    with _db() as con:
        row = con.execute("SELECT * FROM otp WHERE email=? AND purpose=?", (email, purpose)).fetchone()
        if not row or row["expires"] < time.time():
            return False, "The code has expired. Request a new one."
        if row["tries"] >= MAX_OTP_TRIES:
            return False, "Too many wrong attempts. Request a new code."
        good = hmac.compare_digest(row["code_hash"], hashlib.sha256((row["salt"] + code).encode()).hexdigest())
        if not good:
            con.execute("UPDATE otp SET tries=tries+1 WHERE email=? AND purpose=?", (email, purpose))
            return False, "Incorrect code."
        con.execute("DELETE FROM otp WHERE email=? AND purpose=?", (email, purpose))
    return True, "ok"


# ---- accounts -----------------------------------------------------------------------------------------
def register_user(name, email, password, confirm):
    name, email = (name or "").strip(), norm_email(email)
    if len(name) < 2:
        return False, "Please enter your name.", False
    if not EMAIL_RE.match(email):
        return False, "Enter a valid e-mail address.", False
    if ALLOWED_EMAIL_DOMAINS and email.split("@")[1] not in ALLOWED_EMAIL_DOMAINS:
        return False, "Registration is limited to: " + ", ".join(ALLOWED_EMAIL_DOMAINS), False
    if password != confirm:
        return False, "Passwords do not match.", False
    bad = check_password_policy(password)
    if bad:
        return False, bad, False
    with _db() as con:
        row = con.execute("SELECT status FROM users WHERE email=?", (email,)).fetchone()
        if row and row["status"] != "pending":
            return False, "This e-mail is already registered. Please sign in.", False
        n_users = con.execute("SELECT COUNT(*) c FROM users").fetchone()["c"]
        role = "admin" if (email in ADMIN_EMAILS or (not ADMIN_EMAILS and n_users == 0)) else "analyst"
        salt = _new_salt()
        status = "pending" if REQUIRE_EMAIL_VERIFICATION else "active"
        con.execute("REPLACE INTO users(email,name,salt,pw_hash,role,status,created,failed,locked_until) "
                    "VALUES(?,?,?,?,?,?,?,0,0)",
                    (email, name, salt, _hash_pw(password, salt), role, status, time.time()))
    audit(email, "register", role)
    if not REQUIRE_EMAIL_VERIFICATION:
        return True, "Account created. You can sign in now.", False
    ok, msg, _ = _issue_otp(email, "register")
    if not ok:
        return False, msg, False
    return True, msg + " Enter it below to activate your account.", True


def resend_code(email, purpose="register"):
    email = norm_email(email)
    with _db() as con:
        row = con.execute("SELECT status FROM users WHERE email=?", (email,)).fetchone()
    if not row or (purpose == "register" and row["status"] != "pending"):
        return False, "No pending account for this e-mail."
    ok, msg, _ = _issue_otp(email, purpose)
    return ok, msg


def activate_account(email, code):
    email = norm_email(email)
    ok, msg = verify_otp(email, code, "register")
    if not ok:
        return False, msg
    with _db() as con:
        con.execute("UPDATE users SET status='active' WHERE email=? AND status='pending'", (email,))
    audit(email, "verify_email", "")
    return True, "E-mail verified. You can now sign in."


def login(email, password):
    email = norm_email(email)
    ev, err = None, None
    with _db() as con:
        u = con.execute("SELECT * FROM users WHERE email=?", (email,)).fetchone()
        if not u:
            _hash_pw(password or "x", "00" * 16)          # keep timing similar for unknown users
            ev, err = ("login_failed", "unknown user"), "Invalid e-mail or password."
        elif u["locked_until"] > time.time():
            mins = int((u["locked_until"] - time.time()) // 60) + 1
            err = f"Too many failed attempts. Try again in about {mins} min."
        elif not hmac.compare_digest(u["pw_hash"], _hash_pw(password or "", u["salt"])):
            failed = u["failed"] + 1
            lock = time.time() + LOCK_MINUTES * 60 if failed >= MAX_FAILED_LOGINS else 0
            con.execute("UPDATE users SET failed=?, locked_until=? WHERE email=?",
                        (0 if lock else failed, lock, email))
            ev, err = ("login_failed", f"attempt {failed}"), "Invalid e-mail or password."
        elif u["status"] == "pending":
            err = "Please verify your e-mail first (use the Create account tab and the code we sent)."
        elif u["status"] == "disabled":
            err = "This account has been disabled. Contact an administrator."
        else:
            con.execute("UPDATE users SET failed=0, locked_until=0, last_login=? WHERE email=?",
                        (time.time(), email))
    if ev:
        audit(email, *ev)
    if err:
        return None, err
    token = secrets.token_urlsafe(32)
    SESSIONS[token] = {"email": email, "name": u["name"], "role": u["role"],
                       "exp": time.time() + SESSION_HOURS * 3600}
    audit(email, "login", "")
    return token, "ok"


def get_session(token):
    s = SESSIONS.get(token or "")
    if not s:
        return None
    if s["exp"] < time.time():
        SESSIONS.pop(token, None)
        return None
    return s


def logout(token):
    s = SESSIONS.pop(token or "", None)
    if s:
        audit(s["email"], "logout", "")


def request_reset(email):
    email = norm_email(email)
    generic = "If this e-mail has an account, a reset code has been sent."
    with _db() as con:
        row = con.execute("SELECT status FROM users WHERE email=?", (email,)).fetchone()
    if row and row["status"] == "active":
        ok, msg, _ = _issue_otp(email, "reset")
        audit(email, "reset_requested", "sent" if ok else "failed")
        if DEV_SHOW_OTP or not ok and not email_configured():
            return ok, msg
    return True, generic


def reset_password(email, code, new_pw, confirm):
    email = norm_email(email)
    if new_pw != confirm:
        return False, "Passwords do not match."
    bad = check_password_policy(new_pw)
    if bad:
        return False, bad
    ok, msg = verify_otp(email, code, "reset")
    if not ok:
        return False, msg
    salt = _new_salt()
    with _db() as con:
        con.execute("UPDATE users SET salt=?, pw_hash=?, failed=0, locked_until=0 WHERE email=?",
                    (salt, _hash_pw(new_pw, salt), email))
    for t in [t for t, s in SESSIONS.items() if s["email"] == email]:
        SESSIONS.pop(t, None)
    audit(email, "password_reset", "")
    return True, "Password updated. You can sign in with the new password."


# ---- administration -----------------------------------------------------------------------------------
def list_users():
    with _db() as con:
        return [dict(r) for r in con.execute(
            "SELECT email,name,role,status,created,last_login FROM users ORDER BY created")]


def recent_audit(n=40):
    with _db() as con:
        return [dict(r) for r in con.execute("SELECT ts,email,action,detail FROM audit ORDER BY id DESC LIMIT ?", (n,))]


def admin_set(admin_email, target, field, value):
    target = norm_email(target)
    with _db() as con:
        t = con.execute("SELECT * FROM users WHERE email=?", (target,)).fetchone()
        if not t:
            return False, "No such user."
        if field in ("status", "role") and target == admin_email and value != t[field]:
            return False, "You cannot change your own role or status."
        con.execute(f"UPDATE users SET {field}=? WHERE email=?", (value, target))
    if field == "status" and value == "disabled":
        for tk in [tk for tk, s in SESSIONS.items() if s["email"] == target]:
            SESSIONS.pop(tk, None)
    audit(admin_email, f"admin_{field}", f"{target} -> {value}")
    return True, f"{target}: {field} set to {value}."


def fmt_ts(ts):
    return "-" if not ts else _dt.datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M")


print("Auth backend ready | e-mail delivery configured:", email_configured(), "| database:", AUTH_DB)

In [ ]:
# Optional: confirm e-mail delivery works (sends a test message to your own address)
if email_configured():
    print(send_email(_smtp_creds()[0], "Test from Email Forensics Workbench", "Delivery works."))
else:
    print("E-mail not configured yet - add SMTP_USER and SMTP_PASSWORD in Colab Secrets (or set DEV_SHOW_OTP = True to test).")

In [ ]:
# ======================= Professional front end - part 1: helpers and back-end functions =======================
import gradio as gr
import contextlib
import html as _h
import datetime as _dt

LEVEL_COLOR = {"HIGH": "#ef4444", "MEDIUM": "#f59e0b", "LOW": "#22c55e"}


def ui_esc(x):
    return _h.escape(str(x))


def ui_badge(text, color="#3b82f6"):
    return (f"<span class='ef-badge' style='background:{color}22;color:{color};"
            f"border:1px solid {color}66'>{ui_esc(text)}</span>")


def ui_chip(text, kind="neutral"):
    return f"<span class='ef-chip ef-{kind}'>{ui_esc(text)}</span>"


def ui_auth_chip(name, val):
    kind = "good" if val == "pass" else "neutral" if val == "missing" else "bad"
    return ui_chip(f"{name} {val}", kind)


def ui_gauge(pct, level):
    color, arc = LEVEL_COLOR[level], 157.08
    dash = arc * pct / 100
    return (
        "<svg viewBox='0 0 120 72' width='230' height='138'>"
        "<path d='M10 62 A50 50 0 0 1 110 62' fill='none' stroke='#22304d' "
        "stroke-width='11' stroke-linecap='round'/>"
        f"<path d='M10 62 A50 50 0 0 1 110 62' fill='none' stroke='{color}' stroke-width='11' "
        f"stroke-linecap='round' stroke-dasharray='{dash:.1f} {arc}'/>"
        f"<text x='60' y='52' text-anchor='middle' font-size='24' font-weight='700' "
        f"fill='#e6edf7'>{pct}</text>"
        "<text x='60' y='68' text-anchor='middle' font-size='7' fill='#8da2c0'>RISK SCORE / 100</text>"
        "</svg>")


def ui_bar(label, value, maxv, color):
    w = 0 if maxv == 0 else max(0, min(100, 100 * value / maxv))
    return (f"<div class='ef-barrow'><span>{ui_esc(label)}</span>"
            f"<div class='ef-bar'><div style='width:{w:.0f}%;background:{color}'></div></div>"
            f"<b>{value:.0f}/{maxv}</b></div>")


def ui_risk(p_mal, rule, header, obf_flag):
    parts = {"Classifier": (60 * p_mal, 60), "Content rules": (15 * min(rule / 8, 1), 15),
             "Header checks": (15 * min(header / 6, 1), 15), "Obfuscation": (10 if obf_flag else 0, 10)}
    pct = int(round(sum(v for v, _ in parts.values())))
    level = "HIGH" if pct >= 70 else "MEDIUM" if pct >= 40 else "LOW"
    return pct, level, parts


def ui_table(headers, rows):
    th = "".join(f"<th>{ui_esc(x)}</th>" for x in headers)
    body = "".join("<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>" for r in rows)
    return f"<div class='ef-scroll'><table class='ef-table'><thead><tr>{th}</tr></thead><tbody>{body}</tbody></table></div>"


def looks_like_eml(t):
    head = t[:3000].lower()
    return ("from:" in head) and ("subject:" in head)


def highlight_html(text):
    words, scores = explain(text, show=False)
    mx = max(float(np.abs(scores).max()), 1e-6)
    spans = []
    for w, s in zip(words, scores):
        a = min(abs(float(s)) / mx, 1) * 0.75
        col = f"rgba(239,68,68,{a:.2f})" if s > 0 else f"rgba(59,130,246,{a:.2f})"
        spans.append(f"<span style='background:{col};padding:2px 4px;border-radius:4px'>{ui_esc(w)}</span>")
    legend = ("<div class='ef-muted' style='margin-bottom:8px'>Red = pushes toward phishing, "
              "blue = toward legitimate (word-occlusion, approximate).</div>")
    return f"<div class='ef-card'>{legend}<div style='line-height:2.1'>{' '.join(spans)}</div></div>"


# ------------------------------------------------------------------ tab 1: analyse one email
def analyze_email_pro(raw, redact_pii=True):
    raw = (raw or "").strip()
    if not raw:
        empty = "<div class='ef-card ef-muted'>Paste an email on the left and press <b>Analyze</b>.</div>"
        return empty, "", "", "", "", None
    h, header_risk, body, sender = None, 0, raw, None
    if looks_like_eml(raw):
        try:
            h = analyze_headers(raw)
            part = h["msg"].get_body(preferencelist=("plain", "html"))
            text = part.get_content() if part else ""
            body = f"Subject: {h['subject']} | {text}"
            sender, header_risk = h["from"], h["header_risk"]
        except Exception:
            h = None
    obf = analyze_obfuscation(body)
    clean = normalize_text(clean_text(body))
    pii = {}
    if redact_pii:
        clean, pii = redact(clean)
    rep = forensic_report(clean, sender, k=4)
    cls, ind = rep["classification"], rep["indicators"]
    obf_flag = bool(obf["zero_width"] or obf["mixed_script_tokens"])
    pct, level, parts = ui_risk(cls["p_malicious"], ind["rule_risk_score"], header_risk, obf_flag)
    color = LEVEL_COLOR[level]
    mal = cls["p_malicious"] > 0.5

    chips = [ui_chip(f"{ind['num_urls']} URL(s)", "warn" if ind["num_urls"] else "neutral")]
    if ind["ip_based_url"]:
        chips.append(ui_chip("IP-based link", "bad"))
    if ind["url_shortener"]:
        chips.append(ui_chip("URL shortener", "bad"))
    chips += [ui_chip(f"urgency: {t}", "warn") for t in ind["urgency_terms"][:4]]
    chips += [ui_chip(f"credential: {t}", "bad") for t in ind["credential_terms"][:4]]
    chips += [ui_chip(f"financial: {t}", "warn") for t in ind["financial_terms"][:3]]
    if obf["zero_width"]:
        chips.append(ui_chip(f"{obf['zero_width']} hidden zero-width chars", "bad"))
    if obf["mixed_script_tokens"]:
        chips.append(ui_chip("lookalike (homoglyph) letters", "bad"))
    if h:
        chips += [ui_auth_chip(k.upper(), h["auth"][k]) for k in ("spf", "dkim", "dmarc")]
    bars = "".join(ui_bar(k, v, m, color) for k, (v, m) in parts.items())
    pii_txt = ("PII redacted before analysis: " + ", ".join(f"{k}={v}" for k, v in pii.items())
               if pii else ("No PII found." if redact_pii else "PII redaction is off."))
    result = (
        "<div class='ef-card'><div class='ef-result'>"
        f"<div class='ef-gauge'>{ui_gauge(pct, level)}</div>"
        "<div class='ef-grow'>"
        f"<div>{ui_badge(level + ' RISK', color)} "
        f"{ui_badge('PHISHING / SPAM' if mal else 'LEGITIMATE', '#ef4444' if mal else '#22c55e')}</div>"
        f"<div class='ef-big'>p(malicious) = {cls['p_malicious']:.3f}</div>"
        f"<div class='ef-barwrap'>{bars}</div></div></div>"
        f"<div class='ef-chips'>{''.join(chips)}</div>"
        f"<div class='ef-muted' style='margin-top:10px'>{ui_esc(pii_txt)}</div></div>")

    if h:
        hops = " &rarr; ".join(ui_esc(",".join(x["ips"]) or "?") for x in h["hops"]) or "n/a"
        flags = "".join(f"<li>{ui_esc(f)}</li>" for f in h["flags"]) or "<li>No header red flags.</li>"
        rows = [["From", ui_esc(h["from"])], ["Reply-To", ui_esc(h["reply_to"] or "-")],
                ["Return-Path", ui_esc(h["return_path"] or "-")], ["Subject", ui_esc(h["subject"])],
                ["Authentication", " ".join(ui_auth_chip(k.upper(), h["auth"][k]) for k in ("spf", "dkim", "dmarc"))],
                ["Received chain (origin first)", hops]]
        evidence = (f"<div class='ef-card'>{ui_table(['Field', 'Value'], rows)}"
                    f"<h4>Header findings</h4><ul class='ef-list'>{flags}</ul></div>")
    else:
        evidence = ("<div class='ef-card ef-muted'>Plain-text input: no headers to analyse. "
                    "Paste a raw .eml (with headers) for SPF/DKIM/DMARC and relay-chain checks.</div>")

    try:
        hl = highlight_html(clean)
    except Exception as e:
        hl = f"<div class='ef-card ef-muted'>Explanation unavailable: {ui_esc(e)}</div>"

    sim_rows = [[ui_badge("MALICIOUS", "#ef4444") if s["label"] else ui_badge("LEGIT", "#22c55e"),
                 f"{s['similarity']:.2f}", ui_esc(s["snippet"][:220])] for s in rep["similar_emails"]]
    sims = f"<div class='ef-card'>{ui_table(['Known label', 'Similarity', 'Excerpt'], sim_rows)}</div>"

    stamp = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs(f"{OUT_DIR}/reports", exist_ok=True)
    path = f"{OUT_DIR}/reports/analysis_{stamp}.md"
    with open(path, "w") as f:
        f.write(f"# Email forensic analysis ({stamp})\n\nRisk: {level} ({pct}/100)\n"
                f"Verdict: {cls['label']} (p={cls['p_malicious']:.3f})\n\n## Report\n{rep['report']}\n")
    return result, evidence, hl, sims, rep["report"], path


# ------------------------------------------------------------------ tab 2: case investigation
def ui_timeline(timeline):
    out = []
    for k, events in enumerate(timeline, 1):
        items = []
        for when, kind, obj in events:
            if kind == "email":
                r = obj
                cls_ = "ef-tl-bad" if r["flags"] else "ef-tl-ok"
                fl = "".join(ui_chip(f, "bad") for f in r["flags"])
                items.append(
                    f"<div class='ef-tl-item {cls_}'><div class='ef-tl-dot'></div><div class='ef-tl-body'>"
                    f"<div><b>{ui_esc(r['date_iso'][:16].replace('T', ' '))}</b> "
                    f"{ui_chip(r['h12'], 'hash')}</div>"
                    f"<div>{ui_esc(r['from_addr'])} &rarr; {ui_esc(', '.join(r['to']))}</div>"
                    f"<div class='ef-muted'>{ui_esc(r['subject'][:80])}</div><div>{fl}</div></div></div>")
            else:
                mid, w = obj
                lo = w["lo"].isoformat()[:16] if w["lo"] else "?"
                hi = w["hi"].isoformat()[:16] if w["hi"] else "?"
                items.append(
                    "<div class='ef-tl-item ef-tl-miss'><div class='ef-tl-dot'></div><div class='ef-tl-body'>"
                    f"<div><b>MISSING MESSAGE</b> {ui_chip(mid, 'warn')}</div>"
                    f"<div class='ef-muted'>Referenced by {ui_esc(', '.join(w['children']))}; "
                    f"expected between {ui_esc(lo)} and {ui_esc(hi)} (possible deletion).</div></div></div>")
        out.append(f"<div class='ef-card'><h4>Thread {k}</h4><div class='ef-tl'>{''.join(items)}</div></div>")
    return "".join(out)


def ui_case_kpis():
    flagged = sum(1 for r in CASE if r["flags"])
    ips = {ip for r in CASE for ip in r["ips"]}
    ok = verify_custody()
    kp = [("Emails", len(CASE), "#3b82f6"), ("Flagged", flagged, "#ef4444" if flagged else "#22c55e"),
          ("Missing messages", len(missing_windows(CASE)), "#f59e0b"), ("Relay IPs", len(ips), "#8b5cf6"),
          ("Custody log", "INTACT" if ok else "BROKEN", "#22c55e" if ok else "#ef4444")]
    cards = "".join(f"<div class='ef-kpi' style='border-top:3px solid {c}'><div class='ef-kv'>{v}</div>"
                    f"<div class='ef-kl'>{ui_esc(k)}</div></div>" for k, v, c in kp)
    cs = find_compromise_start(GRAPH, CASE)
    note = ""
    if cs:
        f = cs["first"]
        note = ("<div class='ef-card ef-alert'><b>Earliest anomalous email:</b> "
                f"{ui_chip(f['h12'], 'hash')} {ui_esc(f['date_iso'][:16].replace('T', ' '))} from "
                f"{ui_esc(f['from_addr'])}. New relay IPs first seen there: "
                f"{ui_esc(', '.join(cs['new_ips']) or 'none')}. Downstream emails: "
                f"{' '.join(ui_chip(r['h12'], 'hash') for r in cs['downstream']) or 'none'}.</div>")
    return f"<div class='ef-kpis'>{cards}</div>{note}"


def load_case_pro(files, use_demo):
    global GRAPH
    CASE.clear(); REC_BY_H.clear(); EVID.clear(); RAW.clear()
    n = 0
    if use_demo:
        n += ingest_mbox(DEMO_MBOX)
    paths = files if isinstance(files, list) else ([files] if files else [])
    for p in [getattr(f, "name", f) for f in paths]:
        low = p.lower()
        if low.endswith((".mbox", ".mbx")):
            n += ingest_mbox(p)
        elif low.endswith(".eml"):
            rec = parse_raw(open(p, "rb").read(), os.path.basename(p))
            CASE.append(rec)
            REC_BY_H[rec["h12"]] = rec
            custody_event("ingest_email", {"sha256": rec["sha"], "source": rec["source"]})
            n += 1
    if not CASE:
        msg = "<div class='ef-card ef-muted'>No emails loaded. Tick the demo case or upload .mbox / .eml files.</div>"
        return msg, "", None
    CASE.sort(key=lambda r: r["date"])
    build_case_index()
    GRAPH = build_graph(CASE)
    tl = thread_timeline(GRAPH, CASE)
    if tl:
        tl_html = ui_timeline(tl)
    else:
        tl_html = ui_timeline([[(r["date"], "email", r) for r in CASE]])
    return ui_case_kpis(), tl_html, draw_graph(GRAPH, show=False)


def investigate_struct(question, k=2, use_llm=True):
    seeds = [r["h12"] for r, *_ in hybrid_search(question, None, k=k)]
    ctx = graph_context(GRAPH, seeds)
    pack = evidence_pack(ctx)
    findings = llm_findings(question, pack) if use_llm else []
    source = "LLM, every statement verified"
    ok, bad = ground_check(findings) if findings else ([], [])
    if not ok:
        source = "Template findings built directly from stored evidence"
        ok, extra = ground_check(deterministic_findings(ctx))
        bad = bad + extra
    return {"ok": ok, "bad": bad, "source": source, "seeds": seeds,
            "ctx": [r["h12"] for r in ctx], "problems": verify_integrity()}


def ask_case_pro(question, use_llm):
    if not CASE:
        return "<div class='ef-card ef-muted'>Load a case first.</div>", ""
    q = question or "How did the compromise start and which emails were affected?"
    res = investigate_struct(q, k=2, use_llm=use_llm)
    items = "".join(
        f"<li>{ui_esc(s)} {''.join(ui_chip(c, 'hash') for c in cs)}</li>" for s, cs, _ in res["ok"])
    answer = (f"<div class='ef-card'><div>{ui_badge(res['source'], '#3b82f6')}</div>"
              f"<h4>Findings</h4><ul class='ef-list'>{items or '<li>No verified findings.</li>'}</ul>"
              f"<div class='ef-muted'>Context emails: {' '.join(ui_chip(h, 'hash') for h in res['ctx'])}</div></div>")
    rej = ""
    if res["bad"]:
        rows = "".join(f"<li>{ui_esc(s[:140])} <span class='ef-muted'>&mdash; {ui_esc(why)}</span></li>"
                       for s, _, why in res["bad"])
        rej = (f"<details class='ef-card'><summary>{len(res['bad'])} statement(s) rejected by the "
               f"guardrail</summary><ul class='ef-list'>{rows}</ul></details>")
    pr = res["problems"]
    integ = (ui_badge("INTEGRITY VERIFIED - all hashes match stored evidence", "#22c55e") if not pr
             else ui_badge("INTEGRITY PROBLEMS: " + "; ".join(pr), "#ef4444"))
    return answer + rej, f"<div class='ef-card'>{integ}</div>"


def lookup_hash(h12):
    h12 = (h12 or "").strip().strip("[]")
    e = EVID.get(h12)
    if not e:
        return "<div class='ef-card ef-muted'>Unknown hash prefix. Paste a 12-character hash from a finding.</div>"
    status = ""
    if e["kind"] == "email_artifact" and e["email"] in RAW:
        okh = sha256_bytes(RAW[e["email"]]) == e["full"]
        status = ui_badge("Recomputed SHA-256 of stored raw email MATCHES" if okh else "HASH MISMATCH",
                          "#22c55e" if okh else "#ef4444")
    return (f"<div class='ef-card'><div>{ui_chip(e['kind'], 'neutral')} {status}</div>"
            f"<h4>Full SHA-256</h4><code class='ef-code'>{ui_esc(e['full'])}</code>"
            f"<h4>Evidence text (after sanitising and PII redaction)</h4>"
            f"<div class='ef-quote'>{ui_esc(e['text'])}</div></div>")


def check_integrity_pro():
    pr = verify_integrity()
    if CASE and not pr:
        return f"<div class='ef-card'>{ui_badge('INTEGRITY VERIFIED', '#22c55e')} {len(RAW)} raw emails, {len(EVID)} evidence items, custody log valid.</div>"
    return f"<div class='ef-card'>{ui_badge('PROBLEMS FOUND' if pr else 'NO CASE LOADED', '#ef4444')} {ui_esc('; '.join(pr))}</div>"


def export_case_pro():
    path = f"{OUT_DIR}/case/case_export.json"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    data = {"emails": [{"h12": r["h12"], "sha256": r["sha"], "date": r["date_iso"],
                        "from": r["from_addr"], "to": r["to"], "flags": r["flags"]} for r in CASE],
            "custody_log": CUSTODY, "custody_valid": verify_custody()}
    with open(path, "w") as f:
        json.dump(data, f, indent=1)
    return path


# ------------------------------------------------------------------ tab 3: model dashboard
def ui_df_html(df):
    return f"<div class='ef-scroll'>{df.to_html(classes='ef-table', index=False, border=0)}</div>"


def dashboard_pro():
    note = "<div class='ef-card ef-muted'>Run the model-comparison and RAG-evaluation cells to fill this in.</div>"
    cmp_html = ui_df_html(cmp.round(4)) if "cmp" in globals() else note
    rag_html = ui_df_html(rag_eval.round(4)) if "rag_eval" in globals() else note
    kp = []
    if "cmp" in globals():
        best = cmp.sort_values("f1", ascending=False).iloc[0]
        kp += [("Best model", best["model"].split(" ")[0], "#3b82f6"), ("F1", f"{best['f1']:.3f}", "#22c55e"),
               ("Accuracy", f"{best['accuracy']:.3f}", "#22c55e")]
    if "kb" in globals():
        kp.append(("Knowledge base", f"{len(kb):,}", "#8b5cf6"))
    if "test_df" in globals():
        kp.append(("Test emails", f"{len(test_df):,}", "#f59e0b"))
    kp.append(("Device", str(DEVICE).upper(), "#64748b"))
    cards = "".join(f"<div class='ef-kpi' style='border-top:3px solid {c}'><div class='ef-kv'>{v}</div>"
                    f"<div class='ef-kl'>{ui_esc(k)}</div></div>" for k, v, c in kp)
    return f"<div class='ef-kpis'>{cards}</div>", cmp_html, rag_html


def confusion_fig():
    if "y_true" not in globals() or "y_pred" not in globals():
        return None
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix as _cm
    fig, ax = plt.subplots(figsize=(4.6, 3.8))
    sns.heatmap(_cm(y_true, y_pred), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["legit", "phish"], yticklabels=["legit", "phish"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("DistilBERT - test confusion matrix")
    fig.tight_layout()
    return fig


def model_fig():
    if "cmp" not in globals():
        return None
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    cmp.set_index("model")[["accuracy", "precision", "recall", "f1"]].plot.bar(ax=ax, rot=12, ylim=(0.5, 1.0))
    ax.set_title("Model comparison (test set)"); ax.set_xlabel("")
    fig.tight_layout()
    return fig


print("Pro front-end functions ready")

In [ ]:
# ======================= Professional front end - part 2: layout, styling and launch =======================
CSS = """<style>
:root{--ef-card:#111a2e;--ef-line:#22304d;--ef-text:#e6edf7;--ef-muted:#8da2c0}
.gradio-container{max-width:1260px !important;margin:0 auto !important}
.ef-hero{background:linear-gradient(135deg,#0b1220 0%,#12274d 55%,#1d4ed8 150%);border-radius:16px;
 padding:26px 30px;display:flex;justify-content:space-between;align-items:center;gap:16px;flex-wrap:wrap}
.ef-hero h1{margin:0;font-size:27px;color:#fff;letter-spacing:.2px}
.ef-hero p{margin:6px 0 0;color:#b9c8e3;font-size:14px}
.ef-pill{display:inline-block;padding:4px 11px;border-radius:999px;font-size:12px;color:#fff;margin:2px 3px;
 background:rgba(255,255,255,.12);border:1px solid rgba(255,255,255,.2)}
.ef-card{background:var(--ef-card);border:1px solid var(--ef-line);border-radius:14px;padding:18px 20px;
 color:var(--ef-text);margin:8px 0}
.ef-card h4{margin:14px 0 8px;color:#fff;font-size:15px}
.ef-muted{color:var(--ef-muted);font-size:13px}
.ef-alert{border-left:4px solid #ef4444}
.ef-result{display:flex;gap:22px;align-items:center;flex-wrap:wrap}
.ef-gauge{flex:0 0 auto}.ef-grow{flex:1;min-width:240px}
.ef-big{font-size:22px;font-weight:700;margin:10px 0;color:#fff}
.ef-barrow{display:grid;grid-template-columns:110px 1fr 54px;gap:10px;align-items:center;font-size:12px;
 color:var(--ef-muted);margin:5px 0}
.ef-barrow b{color:var(--ef-text);text-align:right}
.ef-bar{height:8px;background:#22304d;border-radius:99px;overflow:hidden}
.ef-bar div{height:100%;border-radius:99px}
.ef-badge{display:inline-block;padding:4px 11px;border-radius:8px;font-size:12px;font-weight:700;
 letter-spacing:.4px;margin:2px 4px 2px 0}
.ef-chips{margin-top:14px}
.ef-chip{display:inline-block;padding:3px 9px;border-radius:7px;font-size:12px;margin:3px 4px 3px 0;
 border:1px solid #33425f;color:#cbd8ee;background:#16223b}
.ef-bad{background:#ef444422;border-color:#ef444477;color:#fca5a5}
.ef-warn{background:#f59e0b22;border-color:#f59e0b77;color:#fcd34d}
.ef-good{background:#22c55e22;border-color:#22c55e77;color:#86efac}
.ef-hash{font-family:ui-monospace,Menlo,Consolas,monospace;background:#1e3a8a44;border-color:#3b82f6aa;color:#93c5fd}
.ef-table{width:100%;border-collapse:collapse;font-size:13px;color:var(--ef-text)}
.ef-table th{background:#16223b;color:#9fb3d1;text-align:left;padding:9px 11px;font-weight:600}
.ef-table td{padding:9px 11px;border-top:1px solid var(--ef-line);vertical-align:top}
.ef-scroll{overflow-x:auto}
.ef-list{margin:6px 0 0 18px;padding:0;line-height:1.7;color:var(--ef-text)}
.ef-kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin:10px 0}
.ef-kpi{background:var(--ef-card);border:1px solid var(--ef-line);border-radius:12px;padding:14px 16px}
.ef-kv{font-size:26px;font-weight:800;color:#fff}.ef-kl{font-size:12px;color:var(--ef-muted);margin-top:2px}
.ef-tl{position:relative;margin-left:8px;border-left:2px solid #2a3b5f;padding-left:0}
.ef-tl-item{position:relative;padding:6px 0 14px 22px}
.ef-tl-dot{position:absolute;left:-7px;top:10px;width:12px;height:12px;border-radius:50%;background:#22c55e;
 border:2px solid #111a2e}
.ef-tl-bad .ef-tl-dot{background:#ef4444}.ef-tl-miss .ef-tl-dot{background:#f59e0b}
.ef-tl-body{background:#16223b;border:1px solid var(--ef-line);border-radius:10px;padding:10px 14px;
 color:var(--ef-text);font-size:13px}
.ef-tl-miss .ef-tl-body{border-style:dashed;border-color:#f59e0b88}
.ef-code{display:block;word-break:break-all;background:#0b1220;border:1px solid var(--ef-line);
 padding:10px;border-radius:8px;color:#93c5fd;font-size:12px}
.ef-quote{background:#16223b;border-left:3px solid #3b82f6;padding:10px 14px;border-radius:6px;font-size:13px}
details.ef-card summary{cursor:pointer;color:#fcd34d}
.ef-foot{text-align:center;color:#8da2c0;font-size:12px;padding:14px 0}

.ef-authwrap{max-width:470px !important;margin:30px auto 0 !important}
.ef-brand{text-align:center;padding:10px 0 4px}
.ef-logo{width:58px;height:58px;border-radius:16px;margin:0 auto 12px;display:flex;align-items:center;justify-content:center;
 font-size:28px;background:linear-gradient(135deg,#1d4ed8,#14b8a6);box-shadow:0 8px 24px #1d4ed855}
.ef-brand h1{margin:0;color:#fff;font-size:26px}.ef-brand p{margin:6px 0 0;color:#8da2c0;font-size:14px}
.ef-msg{border:1px solid;border-radius:10px;padding:10px 14px;font-size:13px;margin:6px 0}
.ef-user{display:flex;gap:12px;align-items:center;padding:6px 2px}
.ef-avatar{width:40px;height:40px;border-radius:50%;background:linear-gradient(135deg,#1d4ed8,#14b8a6);color:#fff;
 display:flex;align-items:center;justify-content:center;font-weight:700;font-size:17px}
.ef-user b{color:#fff}.ef-lock{font-size:12px;color:#8da2c0;text-align:center;margin-top:8px}
</style>"""

APP_TITLE = APP_NAME
llm_state = "LLM reports" if globals().get("llm") is not None else "Template reports"
pills = "".join(f"<span class='ef-pill'>{p}</span>" for p in [
    "DistilBERT classifier", "RAG + BM25 + re-ranker", "Hash-cited evidence",
    f"Device: {str(DEVICE).upper()}", llm_state])
HERO = ("<div class='ef-hero'><div><h1>Email Forensics Workbench</h1>"
        "<p>Deep-learning phishing detection, header forensics and evidence-grounded case investigation</p>"
        f"</div><div>{pills}</div></div>")
EMPTY = "<div class='ef-card ef-muted'>Paste an email on the left and press <b>Analyze</b>.</div>"
ABOUT = """<div class='ef-card'><h4>How it works</h4><ul class='ef-list'>
<li><b>Analyze email:</b> text is sanitised (hidden characters, lookalike letters), optional PII redaction, then a
fine-tuned DistilBERT verdict, rule-based indicators, header checks and similar known emails feed a combined risk score.</li>
<li><b>Case investigation:</b> emails are hashed (SHA-256), parsed and linked into an entity graph; threads and deleted
messages are reconstructed; answers must cite hashes and are rejected if values are not in the cited evidence.</li>
<li><b>Risk score (0-100):</b> 60 classifier + 15 content rules + 15 header checks + 10 obfuscation. HIGH &ge; 70, MEDIUM &ge; 40.</li>
</ul><h4>Limitations</h4><ul class='ef-list'>
<li>The demo case is synthetic; the structure encoder is pre-trained on simulated header profiles.</li>
<li>Retrieval precision@k is a proxy metric; word highlighting is approximate.</li>
<li>The guardrail checks hashes and exact values, not the soundness of reasoning. Reports need human review.</li>
<li>Decision support only - not a legal verdict. Do not load sensitive real mail into a shared public link.</li></ul></div>"""
FOOT = "<div class='ef-foot'>Email Forensics Workbench &middot; for investigation support and research use</div>"


AUTH_BRAND = ("<div class='ef-brand'><div class='ef-logo'>&#128737;</div><h1>Email Forensics Workbench</h1>"
              "<p>Sign in to access the investigation platform</p></div>")
LOCK_NOTE = ("<div class='ef-lock'>&#128274; Passwords are salted and hashed (scrypt). "
             "Sessions expire after " + str(SESSION_HOURS) + " hours.</div>")


# ------------------------------------------------------------------ helpers
def _msg(text, kind="info"):
    col = {"ok": "#22c55e", "err": "#ef4444", "info": "#3b82f6"}[kind]
    return (f"<div class='ef-msg' style='border-color:{col}66;background:{col}18;color:#e6edf7'>"
            f"{ui_esc(text)}</div>")


def banner_html(s):
    role = ui_badge(s["role"].upper(), "#8b5cf6" if s["role"] == "admin" else "#3b82f6")
    return (f"<div class='ef-user'><div class='ef-avatar'>{ui_esc(s['name'][:1].upper())}</div>"
            f"<div><b>{ui_esc(s['name'])}</b> {role}<div class='ef-muted'>{ui_esc(s['email'])}</div></div></div>")


def require(token, admin=False):
    s = get_session(token)
    if not s:
        raise gr.Error("Your session has expired. Please sign in again.")
    if admin and s["role"] != "admin":
        raise gr.Error("Administrator access required.")
    return s


def guard(fn, action, admin=False, pass_session=False):
    """Every protected handler re-checks the server-side session before doing anything."""
    def wrapper(token, *args):
        s = require(token, admin)
        audit(s["email"], action, "")
        return fn(s, *args) if pass_session else fn(*args)
    return wrapper


# ------------------------------------------------------------------ auth handlers
def do_login(email, pw):
    token, msg = login(email, pw)
    if not token:
        return "", gr.update(visible=True), gr.update(visible=False), _msg(msg, "err"), "", gr.update(visible=False), ""
    s = get_session(token)
    return (token, gr.update(visible=False), gr.update(visible=True), "", banner_html(s),
            gr.update(visible=(s["role"] == "admin")), "")


def do_logout(token):
    logout(token)
    return ("", gr.update(visible=True), gr.update(visible=False), _msg("You have been signed out.", "info"),
            "", gr.update(visible=False), EMPTY, "", "", "", "")


def do_register(name, email, pw, pw2):
    ok, msg, need = register_user(name, email, pw, pw2)
    return _msg(msg, "ok" if ok else "err"), gr.update(visible=need)


def do_verify(email, code):
    ok, msg = activate_account(email, code)
    return _msg(msg, "ok" if ok else "err"), gr.update(visible=not ok)


def do_resend(email):
    ok, msg = resend_code(email)
    return _msg(msg, "ok" if ok else "err")


def do_reset_request(email):
    ok, msg = request_reset(email)
    return _msg(msg, "ok" if ok else "err")


def do_reset(email, code, pw, pw2):
    ok, msg = reset_password(email, code, pw, pw2)
    return _msg(msg, "ok" if ok else "err")


# ------------------------------------------------------------------ admin handlers
def admin_views():
    rows = [[ui_esc(u["email"]), ui_esc(u["name"]),
             ui_badge(u["role"], "#8b5cf6" if u["role"] == "admin" else "#3b82f6"),
             ui_badge(u["status"], {"active": "#22c55e", "pending": "#f59e0b"}.get(u["status"], "#ef4444")),
             fmt_ts(u["created"]), fmt_ts(u["last_login"])] for u in list_users()]
    users = f"<div class='ef-card'>{ui_table(['E-mail', 'Name', 'Role', 'Status', 'Created', 'Last login'], rows)}</div>"
    arows = [[fmt_ts(a["ts"]), ui_esc(a["email"]), ui_esc(a["action"]), ui_esc(a["detail"])] for a in recent_audit(40)]
    logs = f"<div class='ef-card'>{ui_table(['Time', 'User', 'Action', 'Detail'], arows)}</div>"
    return users, logs


def admin_do(s, target, field, value):
    ok, msg = admin_set(s["email"], target, field, value)
    return (_msg(msg, "ok" if ok else "err"),) + admin_views()


def admin_refresh(s):
    return ("",) + admin_views()


# ------------------------------------------------------------------ layout
kpi0, cmp0, rag0 = dashboard_pro()

with gr.Blocks(title=APP_NAME) as app:
    gr.HTML(CSS)
    sess = gr.State("")

    # ================= sign-in / registration screen
    with gr.Column(visible=True, elem_classes="ef-authwrap") as auth_col:
        gr.HTML(AUTH_BRAND)
        with gr.Tabs():
            with gr.Tab("Sign in"):
                si_email = gr.Textbox(label="E-mail", placeholder="you@company.com")
                si_pw = gr.Textbox(label="Password", type="password")
                si_btn = gr.Button("Sign in", variant="primary")
                si_msg = gr.HTML()
            with gr.Tab("Create account"):
                rg_name = gr.Textbox(label="Full name")
                rg_email = gr.Textbox(label="E-mail", placeholder="you@company.com")
                rg_pw = gr.Textbox(label="Password (8+ characters, letters and digits)", type="password")
                rg_pw2 = gr.Textbox(label="Confirm password", type="password")
                rg_btn = gr.Button("Create account and send code", variant="primary")
                rg_msg = gr.HTML()
                with gr.Column(visible=False) as otp_group:
                    rg_code = gr.Textbox(label="6-digit code from your e-mail")
                    with gr.Row():
                        rg_verify = gr.Button("Verify e-mail", variant="primary")
                        rg_resend = gr.Button("Resend code")
            with gr.Tab("Forgot password"):
                fp_email = gr.Textbox(label="E-mail")
                fp_send = gr.Button("Send reset code")
                fp_code = gr.Textbox(label="6-digit code")
                fp_pw = gr.Textbox(label="New password", type="password")
                fp_pw2 = gr.Textbox(label="Confirm new password", type="password")
                fp_btn = gr.Button("Reset password", variant="primary")
                fp_msg = gr.HTML()
        gr.HTML(LOCK_NOTE)

    # ================= the application (hidden until signed in)
    with gr.Column(visible=False) as app_col:
        with gr.Row():
            banner = gr.HTML()
            logout_btn = gr.Button("Sign out", size="sm")
        gr.HTML(HERO)
        with gr.Tabs():
            with gr.Tab("Analyze email"):
                with gr.Row():
                    with gr.Column(scale=4):
                        inp = gr.Textbox(lines=18, label="Email (plain text, or raw .eml with headers)",
                                         placeholder="Paste the full email source here...")
                        redact_cb = gr.Checkbox(value=True, label="Redact personal data (names, SSN, phone, account numbers) before analysis")
                        with gr.Row():
                            analyze_btn = gr.Button("Analyze", variant="primary")
                            example_btn = gr.Button("Load example")
                            clear_btn = gr.Button("Clear")
                    with gr.Column(scale=6):
                        result = gr.HTML(value=EMPTY)
                with gr.Tabs():
                    with gr.Tab("Evidence"):
                        evidence = gr.HTML()
                    with gr.Tab("Why this verdict"):
                        why = gr.HTML()
                    with gr.Tab("Similar known emails"):
                        sims = gr.HTML()
                    with gr.Tab("Forensic report"):
                        report = gr.Textbox(lines=12, label="Report")
                        dl = gr.File(label="Download report (.md)")

            with gr.Tab("Case investigation"):
                gr.HTML("<div class='ef-card ef-muted'>Load the synthetic demo case or upload your own "
                        ".mbox / .eml files, then reconstruct the timeline and ask evidence-grounded questions.</div>")
                with gr.Row():
                    with gr.Column(scale=4):
                        files = gr.File(label="Upload .mbox or .eml files", file_count="multiple",
                                        file_types=[".mbox", ".eml"])
                        use_demo = gr.Checkbox(value=True, label="Use built-in demo case (synthetic)")
                        load_btn = gr.Button("Load case", variant="primary")
                    with gr.Column(scale=6):
                        case_kpis = gr.HTML()
                with gr.Tabs():
                    with gr.Tab("Timeline"):
                        timeline = gr.HTML()
                    with gr.Tab("Entity graph"):
                        graph_plot = gr.Plot(label="Emails, addresses, IPs and domains")
                    with gr.Tab("Ask the evidence"):
                        q = gr.Textbox(label="Question",
                                       value="How did the compromise start and which emails were affected?")
                        with gr.Row():
                            q1 = gr.Button("Which emails were spoofed?", size="sm")
                            q2 = gr.Button("What bank details were requested?", size="sm")
                            q3 = gr.Button("Which IP addresses were involved?", size="sm")
                        use_llm = gr.Checkbox(value=True, label="Use LLM (answers are verified; falls back to template findings)")
                        ask_btn = gr.Button("Investigate", variant="primary")
                        answer = gr.HTML()
                        integrity = gr.HTML()
                    with gr.Tab("Verify a citation"):
                        hash_in = gr.Textbox(label="Hash prefix (12 characters, e.g. from a finding)")
                        verify_btn = gr.Button("Look up evidence")
                        hash_out = gr.HTML()
                    with gr.Tab("Export"):
                        integ_btn = gr.Button("Verify integrity of all evidence")
                        integ_out = gr.HTML()
                        export_btn = gr.Button("Export case report (JSON with custody log)")
                        export_file = gr.File(label="Case export")

            with gr.Tab("Model dashboard"):
                dash_kpis = gr.HTML(value=kpi0)
                with gr.Row():
                    cm_plot = gr.Plot(value=confusion_fig(), label="Confusion matrix")
                    mc_plot = gr.Plot(value=model_fig(), label="Model comparison")
                gr.HTML("<div class='ef-card'><h4>Model comparison (test set)</h4></div>")
                dash_cmp = gr.HTML(value=cmp0)
                gr.HTML("<div class='ef-card'><h4>Retrieval evaluation (label precision@k is a proxy metric)</h4></div>")
                dash_rag = gr.HTML(value=rag0)
                refresh_btn = gr.Button("Refresh")

            with gr.Tab("Admin", visible=False) as admin_tab:
                gr.HTML("<div class='ef-card ef-muted'>User management and activity log (administrators only).</div>")
                adm_email = gr.Textbox(label="User e-mail")
                with gr.Row():
                    adm_disable = gr.Button("Disable user", size="sm")
                    adm_enable = gr.Button("Enable user", size="sm")
                    adm_promote = gr.Button("Make admin", size="sm")
                    adm_demote = gr.Button("Make analyst", size="sm")
                    adm_refresh = gr.Button("Refresh", size="sm")
                adm_msg = gr.HTML()
                adm_users = gr.HTML()
                adm_audit = gr.HTML()

            with gr.Tab("About"):
                gr.HTML(ABOUT)
        gr.HTML(FOOT)

    # ================= wiring: authentication
    si_outputs = [sess, auth_col, app_col, si_msg, banner, admin_tab, si_pw]
    si_btn.click(do_login, [si_email, si_pw], si_outputs)
    si_pw.submit(do_login, [si_email, si_pw], si_outputs)
    logout_btn.click(do_logout, [sess], [sess, auth_col, app_col, si_msg, banner, admin_tab,
                                         result, evidence, why, sims, answer])
    rg_btn.click(do_register, [rg_name, rg_email, rg_pw, rg_pw2], [rg_msg, otp_group])
    rg_verify.click(do_verify, [rg_email, rg_code], [rg_msg, otp_group])
    rg_resend.click(do_resend, [rg_email], [rg_msg])
    fp_send.click(do_reset_request, [fp_email], [fp_msg])
    fp_btn.click(do_reset, [fp_email, fp_code, fp_pw, fp_pw2], [fp_msg])

    # ================= wiring: protected application (every handler re-checks the session)
    analyze_btn.click(guard(analyze_email_pro, "analyze_email"), [sess, inp, redact_cb],
                      [result, evidence, why, sims, report, dl])
    example_btn.click(lambda: SAMPLE_EML, None, inp)
    clear_btn.click(lambda: ("", EMPTY, "", "", "", "", None), None,
                    [inp, result, evidence, why, sims, report, dl])
    load_btn.click(guard(load_case_pro, "load_case"), [sess, files, use_demo], [case_kpis, timeline, graph_plot])
    ask_btn.click(guard(ask_case_pro, "ask_case"), [sess, q, use_llm], [answer, integrity])
    q1.click(lambda: "Which emails were spoofed and how can you tell?", None, q)
    q2.click(lambda: "What bank account and routing details were requested?", None, q)
    q3.click(lambda: "Which relay IP addresses were involved and in which emails?", None, q)
    verify_btn.click(guard(lookup_hash, "lookup_hash"), [sess, hash_in], hash_out)
    integ_btn.click(guard(check_integrity_pro, "verify_integrity"), [sess], integ_out)
    export_btn.click(guard(export_case_pro, "export_case"), [sess], export_file)
    refresh_btn.click(guard(lambda: (*dashboard_pro(), confusion_fig(), model_fig()), "dashboard"), [sess],
                      [dash_kpis, dash_cmp, dash_rag, cm_plot, mc_plot])

    adm_out = [adm_msg, adm_users, adm_audit]
    adm_disable.click(guard(lambda s, t: admin_do(s, t, "status", "disabled"), "admin", True, True), [sess, adm_email], adm_out)
    adm_enable.click(guard(lambda s, t: admin_do(s, t, "status", "active"), "admin", True, True), [sess, adm_email], adm_out)
    adm_promote.click(guard(lambda s, t: admin_do(s, t, "role", "admin"), "admin", True, True), [sess, adm_email], adm_out)
    adm_demote.click(guard(lambda s, t: admin_do(s, t, "role", "analyst"), "admin", True, True), [sess, adm_email], adm_out)
    adm_refresh.click(guard(admin_refresh, "admin_view", True, True), [sess], adm_out)

app.launch(share=True, show_error=True)

## Part E - Save results
Run this after the app is launched (or any time after sections 9 and 12 have run). It downloads a small zip of the result tables, case files and the structure encoder.

In [ ]:
import shutil, os

if "cmp" in globals():
    cmp.to_csv(f"{OUT_DIR}/model_comparison.csv", index=False)
if "rag_eval" in globals():
    rag_eval.to_csv(f"{OUT_DIR}/rag_eval.csv", index=False)

small = "/content/results_small"
shutil.rmtree(small, ignore_errors=True)
os.makedirs(small)
for f in os.listdir(OUT_DIR):
    p = os.path.join(OUT_DIR, f)
    if f.endswith((".csv", ".pt")):
        shutil.copy(p, small)
if os.path.isdir(f"{OUT_DIR}/case"):
    shutil.copytree(f"{OUT_DIR}/case", f"{small}/case")
shutil.make_archive("/content/results_small", "zip", small)
print("Saved:", sorted(os.listdir(small)))
try:
    from google.colab import files
    files.download("/content/results_small.zip")
except Exception as e:
    print("Zip at /content/results_small.zip", e)